<a href="https://colab.research.google.com/github/EnzoAA004/PFI_MVPTest_Enzo_AImodule/blob/research%2Fpost-e50-absolute-level-orientation-window/notebooks%2Fpost_e50%2F67B1_postE50_absolute_level_orientation_window.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 67B1 · Post-E50 — Absolute Level Orientation + Lumbar Window

**Roadmap:** 67A (SPIDER relative localization, cerrado) -> 67B (RSNA absolute-level anchor smoke,
**cerrado formalmente**, commit congelado `351d9a5ba63b3cfffea91180c0075f778a858e20` en
`research/post-e50-absolute-level-anchor`) -> **67B1 (este notebook)** -> 67C (axial cluster
pairing, futuro, bloqueado).

## Contexto congelado de 67B (no se modifica, solo se cita)

67B cerró con `GATE_A=PASS, GATE_B=YES, GATE_C=PASS, GATE_D=PASS, GATE_E=PASS, GATE_F=PASS,
GATE_G=UNRESOLVED, GATE_H=PARTIAL, GATE_I=PASS`, `overall_decision_67b=PARTIAL`. Checkpoint frozen:
`cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944`.

Smoke real (3 studies validation): case 1 (8 predicted instances) -> ventana absoluta RSNA en ranks
4-8, sequence SAME; case 2 (9 instances) -> ranks 6-2, REVERSED; case 3 (7 instances) -> ranks 3-7,
SAME. **3/3 monotonic up to global reversal, 0/3 MIXED.** Longitudinal-axis nearest: mean=4.21mm,
median=2.60mm, min=0.26mm, max=11.88mm.

**Conclusión de 67B:** el frozen 67A transfiere relative longitudinal disc ordering útil a RSNA,
pero **no** resuelve naming absoluto automático. Quedan 2 grados de libertad sin resolver: **(A)**
orientación craneal/caudal sin GT; **(B)** selección de la ventana de 5 discos contiguos correcta
sin GT. **67B1 existe únicamente para resolver (A) y (B).**

## Objetivo formal de 67B1

Input: relative disc instances del frozen 67A + geometría DICOM + geometría/confidence derivadas
de la segmentación disponibles en inferencia. Output: `L1-L2...L5-S1` **o** `ABSTAIN`, **sin usar
GT de RSNA durante la inferencia**. GT se usa exclusivamente para: (1) construir targets
supervisados en una fase futura explícitamente aprobada (no en esta entrega); (2) evaluar
predicciones **después** de que la decisión de inferencia ya fue tomada. GT **nunca** se usa para
elegir signo de eje, dirección craneal/caudal, ventana de 5 discos, seleccionar instancias, tunear
un threshold case-specific, voltear una secuencia predicha después de ver labels, o reparar una
predicción incorrecta.

## Prohibiciones de esta revisión

NO se toca `main`, Backend, Frontend, Notebook 67, Notebook 67A, Notebook 67B (cerrado), checkpoint
`cf11...`, `AUTOMATIC_DISC_LOCALIZATION_VALIDATED`, `internal_test`, test oficial RSNA, axial
pairing/67C, pathology grading/classifiers. NO se entrena nada en esta entrega (ni siquiera el
fallback supervisado ligero mencionado como diseño futuro). NO commit. NO push.

Rama: `research/post-e50-absolute-level-orientation-window` (creada desde el commit final de 67B:
`351d9a5ba63b3cfffea91180c0075f778a858e20`, rama `research/post-e50-absolute-level-anchor`).
67B1 es **autocontenido**: no importa 67B como módulo, no depende de su kernel, no lo modifica.


In [1]:
# --- Setup: environment detection ---
import hashlib
import json
import os
import re
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

EXECUTION_START = time.time()

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("IN_COLAB:", IN_COLAB)
print("ENVIRONMENT:", "COLAB" if IN_COLAB else "LOCAL")


IN_COLAB: True
ENVIRONMENT: COLAB


## STEP 1 — Mount Google Drive (Colab only)


In [2]:
DRIVE_MOUNTED = False
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_MOUNTED = Path('/content/drive/MyDrive').is_dir()
    print("Drive mounted:", DRIVE_MOUNTED)
else:
    print("Not in Colab: skipping Drive mount. Local execution uses environment variables instead.")


Mounted at /content/drive
Drive mounted: True


## Configuración explícita de sesión Colab

Igual que 67B: sin dependencia funcional de `PFI_MVP/repo`, `PFI_AI_REPO_ROOT`, ni `ai_service`
importado desde Drive. Todo el runtime necesario se embebe explícitamente más abajo.


In [3]:
if IN_COLAB:
    os.environ["PFI_DRIVE_ROOT"] = "/content/drive/MyDrive/PFI_MVP"
    os.environ["PFI_RSNA_ROOT"] = "/content/drive/MyDrive/PFI_MVP/data/RSNA_LUMBAR_DISC"
    os.environ["PFI_POST_E50_SAGITTAL_CHECKPOINT"] = "/content/drive/MyDrive/PFI_MVP/models/final/sagittal_spider_multiclass_final_best_cf11dcc0.pt"
    for key in ("PFI_DRIVE_ROOT", "PFI_RSNA_ROOT", "PFI_POST_E50_SAGITTAL_CHECKPOINT"):
        print(f"{key} = {os.environ[key]}")
else:
    print("Local execution: Colab Drive environment variables not forced.")


PFI_DRIVE_ROOT = /content/drive/MyDrive/PFI_MVP
PFI_RSNA_ROOT = /content/drive/MyDrive/PFI_MVP/data/RSNA_LUMBAR_DISC
PFI_POST_E50_SAGITTAL_CHECKPOINT = /content/drive/MyDrive/PFI_MVP/models/final/sagittal_spider_multiclass_final_best_cf11dcc0.pt


## STEP 2 — Workspace local del notebook


In [4]:
def _find_repo_root_local(start: Path) -> Path | None:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "notebooks").is_dir() and (candidate / "artifacts").exists():
            return candidate
    return None


_local_repo = _find_repo_root_local(Path.cwd())

if IN_COLAB:
    REPO_ROOT = Path("/content/pfi_post_e50_67B1_workspace")
    REPO_ROOT.mkdir(parents=True, exist_ok=True)
    REPO_ROOT_SOURCE = "ephemeral_colab_workspace"
else:
    REPO_ROOT = _local_repo or Path.cwd()
    REPO_ROOT_SOURCE = "auto_detected_local" if _local_repo is not None else "cwd_local"

print("Workspace source:", REPO_ROOT_SOURCE)
print("Workspace root (folder name only, no absolute path persisted):", REPO_ROOT.name)


Workspace source: ephemeral_colab_workspace
Workspace root (folder name only, no absolute path persisted): pfi_post_e50_67B1_workspace


## STEP 3 — Configure `PFI_DRIVE_ROOT` / `PFI_RSNA_ROOT`

Verificación de contenido real (no solo nombre) antes de aceptar `RSNA_ROOT`.


In [5]:
PFI_DRIVE_ROOT_ENV = os.environ.get("PFI_DRIVE_ROOT")
PFI_DRIVE_ROOT = Path(PFI_DRIVE_ROOT_ENV) if PFI_DRIVE_ROOT_ENV else None
print("PFI_DRIVE_ROOT exists:", PFI_DRIVE_ROOT.is_dir() if PFI_DRIVE_ROOT else None)

RSNA_EXPECTED_FILES = ["train.csv", "train_label_coordinates.csv", "train_series_descriptions.csv", "train_images"]
RSNA_OFFICIAL_TEST_MARKERS = ["test_images", "test_series_descriptions.csv", "sample_submission.csv"]


def _rsna_content_report(path: Path) -> dict:
    if not path.is_dir():
        return {"rsna_root_exists": False, "files_found": {}, "official_test_present": False}
    files_found = {name: (path / name).exists() for name in RSNA_EXPECTED_FILES}
    official_test_present = any((path / marker).exists() for marker in RSNA_OFFICIAL_TEST_MARKERS)
    return {"rsna_root_exists": True, "files_found": files_found, "official_test_present": official_test_present}


RSNA_ROOT_ENV = os.environ.get("PFI_RSNA_ROOT")
RSNA_ROOT = Path(RSNA_ROOT_ENV) if RSNA_ROOT_ENV else None
rsna_content_report = _rsna_content_report(RSNA_ROOT) if RSNA_ROOT else {"rsna_root_exists": False, "files_found": {}, "official_test_present": False}

RSNA_AVAILABLE = bool(
    rsna_content_report["rsna_root_exists"]
    and rsna_content_report["files_found"].get("train.csv")
    and rsna_content_report["files_found"].get("train_label_coordinates.csv")
    and rsna_content_report["files_found"].get("train_series_descriptions.csv")
)
OFFICIAL_TEST_PRESENT = bool(rsna_content_report["official_test_present"])
OFFICIAL_TEST_ACCESSED = False  # structurally never set True anywhere in this notebook

print(json.dumps(rsna_content_report, indent=2))
print("RSNA_AVAILABLE:", RSNA_AVAILABLE)
print("officialTestPresent:", OFFICIAL_TEST_PRESENT, "officialTestAccessed:", OFFICIAL_TEST_ACCESSED)
if not RSNA_AVAILABLE:
    print("Expected outside Colab / without Drive mounted -- not a failure by itself.")


PFI_DRIVE_ROOT exists: True
{
  "rsna_root_exists": true,
  "files_found": {
    "train.csv": true,
    "train_label_coordinates.csv": true,
    "train_series_descriptions.csv": true,
    "train_images": true
  },
  "official_test_present": false
}
RSNA_AVAILABLE: True
officialTestPresent: False officialTestAccessed: False


## Privacy helpers, allowed write scope, git identity


In [6]:
def opaque_id(raw_value) -> str:
    return hashlib.sha256(str(raw_value).encode("utf-8")).hexdigest()[:12]


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


ANCHOR_DIR = REPO_ROOT / "artifacts" / "post_e50" / "absolute_level_orientation_window"
REPORT_DIR = REPO_ROOT / "reports" / "post_e50"
ALLOWED_WRITE_ROOTS = [
    REPO_ROOT / "notebooks" / "post_e50",
    REPO_ROOT / "artifacts" / "post_e50",
    REPO_ROOT / "reports" / "post_e50",
]
DRIVE_RESULTS_DIR = (PFI_DRIVE_ROOT / "results" / "post_e50" / "67B1") if PFI_DRIVE_ROOT is not None else None
DRIVE_METRICS_DIR = (PFI_DRIVE_ROOT / "metrics" / "post_e50" / "67B1") if PFI_DRIVE_ROOT is not None else None
DRIVE_FIGURES_DIR = (PFI_DRIVE_ROOT / "figures" / "post_e50" / "67B1") if PFI_DRIVE_ROOT is not None else None
if PFI_DRIVE_ROOT is not None:
    ALLOWED_WRITE_ROOTS.extend([DRIVE_RESULTS_DIR, DRIVE_METRICS_DIR, DRIVE_FIGURES_DIR])

warnings: list[str] = []
limitations: list[str] = []
FORBIDDEN_IDENTIFIER_FIELDS = ("PatientName", "PatientID", "PatientBirthDate", "InstitutionName", "AccessionNumber")


def safe_write_text(path: Path, content: str) -> None:
    path = path.resolve()
    if not any(str(path).startswith(str(root.resolve())) for root in ALLOWED_WRITE_ROOTS if root is not None):
        raise RuntimeError(f"Refusing to write outside allowed trees: {path}")
    if "C:\\Users\\" in content or "/Users/" in content:
        raise RuntimeError(f"Refusing to persist a local filesystem path into {path.name}")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")


def run_git(*args: str) -> str | None:
    try:
        result = subprocess.run(["git", *args], cwd=REPO_ROOT, capture_output=True, text=True, check=True)
        return result.stdout.strip()
    except Exception:
        return None


if IN_COLAB:
    GIT_BRANCH = "ephemeral-colab-workspace"
    GIT_COMMIT = None
else:
    GIT_BRANCH = run_git("branch", "--show-current")
    GIT_COMMIT = run_git("rev-parse", "HEAD")

GENERATED_AT = datetime.now(timezone.utc).isoformat()
BASE_COMMIT_67B = "351d9a5ba63b3cfffea91180c0075f778a858e20"
print("GIT_BRANCH:", GIT_BRANCH)
print("GIT_COMMIT:", GIT_COMMIT)
print("BASE_COMMIT_67B (frozen, cited only):", BASE_COMMIT_67B)


GIT_BRANCH: ephemeral-colab-workspace
GIT_COMMIT: None
BASE_COMMIT_67B (frozen, cited only): 351d9a5ba63b3cfffea91180c0075f778a858e20


## STEP 4 — Dependencias (torch/SimpleITK/scipy/pydicom), comprobación explícita


In [7]:
def _check_importable(module_name: str) -> str | None:
    try:
        module = __import__(module_name)
        return getattr(module, "__version__", "unknown_version")
    except ImportError:
        return None


torch_probe = _check_importable("torch")
sitk_probe = _check_importable("SimpleITK")
scipy_probe = _check_importable("scipy")
pydicom_probe = _check_importable("pydicom")

print("torch:", torch_probe)
print("SimpleITK:", sitk_probe if sitk_probe else "NOT INSTALLED")
print("scipy:", scipy_probe if scipy_probe else "NOT INSTALLED")
print("pydicom:", pydicom_probe if pydicom_probe else "NOT INSTALLED")
NEEDS_SITK = sitk_probe is None
NEEDS_SCIPY = scipy_probe is None
NEEDS_PYDICOM = pydicom_probe is None
if NEEDS_SITK or NEEDS_SCIPY or NEEDS_PYDICOM:
    print("Run the next cell explicitly to install missing dependencies (not automatic).")


torch: 2.11.0+cpu
SimpleITK: NOT INSTALLED
scipy: 1.16.3
pydicom: NOT INSTALLED
Run the next cell explicitly to install missing dependencies (not automatic).


In [8]:
if NEEDS_SITK:
    print("Installing SimpleITK...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "SimpleITK"])
if NEEDS_SCIPY:
    print("Installing scipy...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "scipy"])
if NEEDS_PYDICOM:
    print("Installing pydicom...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "pydicom"])

import torch
import SimpleITK as sitk
import scipy
import pydicom

DEPENDENCY_VERSIONS = {
    "torch_version": torch.__version__,
    "simpleitk_version": sitk.Version_VersionString(),
    "scipy_version": scipy.__version__,
    "pydicom_version": pydicom.__version__,
    "cuda_available": torch.cuda.is_available(),
}
print(json.dumps(DEPENDENCY_VERSIONS, indent=2, default=str))


Installing SimpleITK...
Installing pydicom...
{
  "torch_version": "2.11.0+cpu",
  "simpleitk_version": "2.5.6",
  "scipy_version": "1.16.3",
  "pydicom_version": "3.0.2",
  "cuda_available": false
}


## STAGE A — RSNA dataset structure (GATE A) — inspección real, reusa el esquema ya auditado por 67B

Mismas 3 CSV reales que 67B: `train.csv`, `train_label_coordinates.csv`,
`train_series_descriptions.csv`. No se asume el esquema -- se lee y verifica en cada corrida.


In [9]:
GATE_A_RSNA_dataset_structure = "NOT_RUN"
train_df = pd.DataFrame()
coords_df = pd.DataFrame()
series_desc_df = pd.DataFrame()
csv_schema_report = {}

if RSNA_AVAILABLE:
    try:
        train_df = pd.read_csv(RSNA_ROOT / "train.csv", dtype={"study_id": "string"})
        coords_df = pd.read_csv(RSNA_ROOT / "train_label_coordinates.csv", dtype={"study_id": "string", "series_id": "string"})
        series_desc_df = pd.read_csv(RSNA_ROOT / "train_series_descriptions.csv", dtype={"study_id": "string", "series_id": "string"})
        GATE_A_RSNA_dataset_structure = "PASS"
    except Exception as exc:
        GATE_A_RSNA_dataset_structure = "FAIL"
        warnings.append(f"RSNA CSV read failed: {exc}")
else:
    warnings.append("STAGE A NOT_RUN: RSNA dataset not available in this run.")

if GATE_A_RSNA_dataset_structure == "PASS":
    csv_schema_report = {
        "shapes": {"train.csv": list(train_df.shape), "train_label_coordinates.csv": list(coords_df.shape), "train_series_descriptions.csv": list(series_desc_df.shape)},
        "csv_sha256": {
            "train.csv": sha256_file(RSNA_ROOT / "train.csv"),
            "train_label_coordinates.csv": sha256_file(RSNA_ROOT / "train_label_coordinates.csv"),
            "train_series_descriptions.csv": sha256_file(RSNA_ROOT / "train_series_descriptions.csv"),
        },
    }
    print(json.dumps(csv_schema_report, indent=2, default=str))
print()
print("GATE_A_RSNA_dataset_structure:", GATE_A_RSNA_dataset_structure)


{
  "shapes": {
    "train.csv": [
      1975,
      26
    ],
    "train_label_coordinates.csv": [
      48692,
      7
    ],
    "train_series_descriptions.csv": [
      6294,
      3
    ]
  },
  "csv_sha256": {
    "train.csv": "f0c9e06486bcddcd83b1ee1d95ab9db8e028d7c3c9fcbada950f0b8e4d828528",
    "train_label_coordinates.csv": "416fb434b5bdbf69814210d03dd791dff5ef9a010bac004f5f68e8b79e852635",
    "train_series_descriptions.csv": "bf8cc1aa55e4b5536b2f0fa8a060fc1912faeb65f9ba422d9d91b92a947b1c33"
  }
}

GATE_A_RSNA_dataset_structure: PASS


## Level normalization + series classification (reusa lógica ya validada en 67B)


In [10]:
CANONICAL_LEVELS = ["L1-L2", "L2-L3", "L3-L4", "L4-L5", "L5-S1"]


def normalize_level_value(raw) -> str | None:
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return None
    text = str(raw).strip().upper().replace("/", "-").replace("_", "-")
    return text if text in CANONICAL_LEVELS else None


def classify_series_description(raw: str) -> str:
    text = str(raw).strip().lower()
    if "t2" in text and ("stir" in text or "sag" in text):
        if "sag" in text:
            return "sagittal_t2_stir"
    if "sag" in text and "t1" in text:
        return "sagittal_t1"
    if "sag" in text and "t2" in text:
        return "sagittal_t2_stir"
    if "ax" in text and "t2" in text:
        return "axial_t2"
    return "unclassified"


# Synthetic self-tests (independent of RSNA availability):
assert normalize_level_value("L4/L5") == "L4-L5", "normalize_level_value self-test FAILED"
assert normalize_level_value("not_a_level") is None, "normalize_level_value self-test FAILED (unrecognized)"
assert classify_series_description("Sagittal T2/STIR") == "sagittal_t2_stir", "classify_series_description self-test FAILED"
assert classify_series_description("Sagittal T1") == "sagittal_t1", "classify_series_description self-test FAILED"
print("normalize_level_value / classify_series_description: synthetic self-tests PASSED.")

level_column_present = False
condition_column_present = False
PRIMARY_REFERENCE_CONDITION = "Spinal Canal Stenosis"
PRIMARY_REFERENCE_SEQUENCE_ROLE = "sagittal_t2_stir"

if GATE_A_RSNA_dataset_structure == "PASS" and len(coords_df):
    level_column_present = "level" in coords_df.columns
    condition_column_present = "condition" in coords_df.columns
    if level_column_present:
        coords_df = coords_df.copy()
        coords_df["level_raw"] = coords_df["level"]
        coords_df["level_normalized"] = coords_df["level_raw"].apply(normalize_level_value)

print("level_column_present:", level_column_present, "condition_column_present:", condition_column_present)


normalize_level_value / classify_series_description: synthetic self-tests PASSED.
level_column_present: True condition_column_present: True


## GATE C — split leakage audit + reproducible study_id-level split

Mismo principio que 65/66/67A/67B: nunca separar contenido de un mismo `study_id` entre
train/validation/internal_test. `RSNA_INTERNAL_TEST_LOCKED=True` bloquea estructuralmente el uso
del holdout interno y del test oficial.


In [11]:
def compute_study_level_leakage_audit(train_ids: set, validation_ids: set, holdout_ids: set) -> dict:
    train_val_overlap = train_ids & validation_ids
    train_holdout_overlap = train_ids & holdout_ids
    val_holdout_overlap = validation_ids & holdout_ids
    return {
        "train_validation_overlap_count": len(train_val_overlap),
        "train_holdout_overlap_count": len(train_holdout_overlap),
        "validation_holdout_overlap_count": len(val_holdout_overlap),
        "leakage_free": len(train_val_overlap) == 0 and len(train_holdout_overlap) == 0 and len(val_holdout_overlap) == 0,
    }


def build_reproducible_study_split(study_ids: list, seed: int = 2026, train_frac: float = 0.7, validation_frac: float = 0.15) -> dict:
    ordered = sorted(study_ids)
    rng = np.random.default_rng(seed)
    shuffled = list(ordered)
    rng.shuffle(shuffled)
    n = len(shuffled)
    n_train = int(n * train_frac)
    n_val = int(n * validation_frac)
    return {
        "train": sorted(shuffled[:n_train]),
        "validation": sorted(shuffled[n_train:n_train + n_val]),
        "internal_test": sorted(shuffled[n_train + n_val:]),
    }


# Synthetic self-tests (independent of RSNA availability):
_clean = compute_study_level_leakage_audit({"a", "b"}, {"c", "d"}, {"e"})
assert _clean["leakage_free"] is True, "compute_study_level_leakage_audit self-test FAILED"
_synthetic_ids = [f"study_{i}" for i in range(20)]
_split_a = build_reproducible_study_split(_synthetic_ids)
_split_b = build_reproducible_study_split(_synthetic_ids)
assert _split_a == _split_b, "build_reproducible_study_split self-test FAILED (not deterministic)"
_all_assigned = set(_split_a["train"]) | set(_split_a["validation"]) | set(_split_a["internal_test"])
assert _all_assigned == set(_synthetic_ids), "build_reproducible_study_split self-test FAILED (coverage)"
print("compute_study_level_leakage_audit / build_reproducible_study_split: synthetic self-tests PASSED.")

RSNA_INTERNAL_TEST_LOCKED = True
GATE_C_split_leakage = "NOT_RUN"
real_split = {"train": [], "validation": [], "internal_test": []}

if GATE_A_RSNA_dataset_structure == "PASS" and "study_id" in series_desc_df.columns:
    all_study_ids = list(set(series_desc_df["study_id"].dropna().unique()))
    real_split = build_reproducible_study_split(all_study_ids)
    audit_result = compute_study_level_leakage_audit(set(real_split["train"]), set(real_split["validation"]), set(real_split["internal_test"]))
    GATE_C_split_leakage = "PASS" if audit_result["leakage_free"] else "FAIL"
    print(json.dumps({"leakage_audit": audit_result, "split_counts": {k: len(v) for k, v in real_split.items()}}, indent=2))
else:
    warnings.append("GATE_C split leakage audit NOT_RUN: GATE_A did not PASS in this run.")

print("RSNA_INTERNAL_TEST_LOCKED:", RSNA_INTERNAL_TEST_LOCKED, "(internal_test never opened/used in this notebook)")
print("GATE_C_split_leakage:", GATE_C_split_leakage)


compute_study_level_leakage_audit / build_reproducible_study_split: synthetic self-tests PASSED.
{
  "leakage_audit": {
    "train_validation_overlap_count": 0,
    "train_holdout_overlap_count": 0,
    "validation_holdout_overlap_count": 0,
    "leakage_free": true
  },
  "split_counts": {
    "train": 1382,
    "validation": 296,
    "internal_test": 297
  }
}
RSNA_INTERNAL_TEST_LOCKED: True (internal_test never opened/used in this notebook)
GATE_C_split_leakage: PASS


## Sagittal T2/STIR + Spinal Canal Stenosis — primary reference table (misma hipótesis auditada en 67B)


In [12]:
sag_t2_canal_stenosis_reference_rows = []
sag_t2_canal_stenosis_audit = {"studies_with_sag_t2_series": 0, "studies_with_canal_stenosis_coords": 0, "studies_with_both": 0}

if GATE_A_RSNA_dataset_structure == "PASS" and level_column_present and "series_description" in series_desc_df.columns:
    roles_all = series_desc_df["series_description"].apply(classify_series_description)
    sag_t2_rows = series_desc_df.assign(sequence_role=roles_all)
    sag_t2_rows = sag_t2_rows[sag_t2_rows["sequence_role"] == PRIMARY_REFERENCE_SEQUENCE_ROLE]
    sag_t2_series_by_study = sag_t2_rows.groupby("study_id")["series_id"].apply(list)

    canal_coords = coords_df[coords_df.get("condition", pd.Series(dtype=object)) == PRIMARY_REFERENCE_CONDITION] if "condition" in coords_df.columns else pd.DataFrame()

    sag_t2_canal_stenosis_audit["studies_with_sag_t2_series"] = int(sag_t2_rows["study_id"].nunique())
    sag_t2_canal_stenosis_audit["studies_with_canal_stenosis_coords"] = int(canal_coords["study_id"].nunique()) if len(canal_coords) else 0
    common_studies = set(sag_t2_rows["study_id"]) & set(canal_coords["study_id"]) if len(canal_coords) else set()
    sag_t2_canal_stenosis_audit["studies_with_both"] = len(common_studies)

    for study_id in sorted(common_studies):
        sag_series_ids = set(sag_t2_series_by_study.get(study_id, []))
        study_canal_coords = canal_coords[(canal_coords["study_id"] == study_id) & (canal_coords["series_id"].isin(sag_series_ids))]
        levels_seen = set()
        for _, r in study_canal_coords.iterrows():
            level_norm = r.get("level_normalized")
            status = "ok"
            if level_norm is None:
                status = "unrecognized_level"
            elif level_norm in levels_seen:
                status = "duplicate_level"
            levels_seen.add(level_norm)
            sag_t2_canal_stenosis_reference_rows.append({
                "study_id_opaque": opaque_id(study_id), "sag_t2_series_id_opaque": opaque_id(r.get("series_id")),
                "level_raw": r.get("level_raw"), "level_normalized": level_norm,
                "instance_number": r.get("instance_number"), "x": r.get("x"), "y": r.get("y"),
                "status": status,
            })
        missing_levels = set(CANONICAL_LEVELS) - {lv for lv in levels_seen if lv is not None}
        for missing in sorted(missing_levels):
            sag_t2_canal_stenosis_reference_rows.append({
                "study_id_opaque": opaque_id(study_id), "sag_t2_series_id_opaque": None,
                "level_raw": None, "level_normalized": missing, "instance_number": None, "x": None, "y": None,
                "status": "missing_level",
            })
else:
    warnings.append("Sagittal T2/STIR + Spinal Canal Stenosis reference table NOT_RUN: GATE_A did not PASS or required columns missing.")

print(json.dumps(sag_t2_canal_stenosis_audit, indent=2, default=str))


{
  "studies_with_sag_t2_series": 1974,
  "studies_with_canal_stenosis_coords": 1974,
  "studies_with_both": 1974
}


## Geometría DICOM real — `dicom_pixel_to_patient_xyz` (misma fórmula DICOM que 66/67/67A/67B)


In [13]:
def dicom_pixel_to_patient_xyz(column_index: float, row_index: float, pixel_spacing: tuple[float, float],
                                image_position_patient: tuple[float, float, float],
                                image_orientation_patient: tuple[float, float, float, float, float, float]) -> np.ndarray:
    # PS3.3 C.7.6.2.1.1. RSNA convention: x=column_index, y=row_index.
    row_cosines = np.array(image_orientation_patient[0:3], dtype=np.float64)
    column_cosines = np.array(image_orientation_patient[3:6], dtype=np.float64)
    origin = np.array(image_position_patient, dtype=np.float64)
    row_spacing, column_spacing = pixel_spacing
    return origin + column_index * column_spacing * row_cosines + row_index * row_spacing * column_cosines


def resolve_dicom_instance_path(train_images_root: Path, study_id: str, series_id: str, instance_number: int) -> Path | None:
    series_dir = train_images_root / str(study_id) / str(series_id)
    if not series_dir.is_dir():
        return None
    candidate = series_dir / f"{instance_number}.dcm"
    if candidate.is_file():
        return candidate
    matches = list(series_dir.glob(f"*{instance_number}*.dcm"))
    return matches[0] if matches else None


def check_xy_inside_frame(column_index: float, row_index: float, rows: int, columns: int) -> bool:
    return (0 <= column_index < columns) and (0 <= row_index < rows)


def check_geometry_finite(*values: float) -> bool:
    return all(np.isfinite(v) for v in values)


def check_orientation_valid(image_orientation_patient: tuple) -> bool:
    if len(image_orientation_patient) != 6:
        return False
    row_cosines = np.array(image_orientation_patient[0:3], dtype=np.float64)
    column_cosines = np.array(image_orientation_patient[3:6], dtype=np.float64)
    return bool(abs(np.linalg.norm(row_cosines) - 1.0) < 1e-2 and abs(np.linalg.norm(column_cosines) - 1.0) < 1e-2 and abs(float(np.dot(row_cosines, column_cosines))) < 1e-2)


# Synthetic self-tests (independent of RSNA availability):
_identity_xyz = dicom_pixel_to_patient_xyz(10.0, 20.0, (1.0, 1.0), (0.0, 0.0, 0.0), (1.0, 0.0, 0.0, 0.0, 1.0, 0.0))
assert np.allclose(_identity_xyz, [10.0, 20.0, 0.0]), "dicom_pixel_to_patient_xyz self-test FAILED (identity)"
_sitk_check_image = sitk.Image(64, 64, 1, sitk.sitkUInt8)
_sitk_check_image.SetSpacing((0.5, 0.7, 1.0))
_sitk_check_image.SetOrigin((100.0, -50.0, 25.0))
_sitk_check_image.SetDirection((0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0))
_sitk_xyz = np.array(_sitk_check_image.TransformContinuousIndexToPhysicalPoint((5.0, 7.0, 0.0)))
_manual_xyz = dicom_pixel_to_patient_xyz(5.0, 7.0, (0.7, 0.5), (100.0, -50.0, 25.0), (0.0, 1.0, 0.0, 1.0, 0.0, 0.0))
assert np.allclose(_sitk_xyz, _manual_xyz, atol=1e-6), "dicom_pixel_to_patient_xyz self-test FAILED (SimpleITK cross-check)"
assert check_xy_inside_frame(10, 10, 64, 64) is True and check_xy_inside_frame(-1, 10, 64, 64) is False, "check_xy_inside_frame self-test FAILED"
assert check_geometry_finite(1.0, 2.0) is True and check_geometry_finite(1.0, float("nan")) is False, "check_geometry_finite self-test FAILED"
assert check_orientation_valid((1.0, 0.0, 0.0, 0.0, 1.0, 0.0)) is True, "check_orientation_valid self-test FAILED"
print("dicom_pixel_to_patient_xyz (cross-validated against SimpleITK) / check_xy_inside_frame / check_geometry_finite / check_orientation_valid: synthetic self-tests PASSED.")


dicom_pixel_to_patient_xyz (cross-validated against SimpleITK) / check_xy_inside_frame / check_geometry_finite / check_orientation_valid: synthetic self-tests PASSED.


## STEP -- Fast path: reuse 67B's validated real-Colab geometry evidence (opcional, auditable)

67B ya ejecuto una auditoria geometrica real completa sobre 2521 instancias DICOM unicas (ver
`351d9a5ba63b3cfffea91180c0075f778a858e20`, notebook 67B, CIERRE FORMAL). Esa evidencia describe
las MISMAS 3 CSVs de RSNA (train.csv, train_label_coordinates.csv, train_series_descriptions.csv).

Si los fingerprints SHA-256 y los counts primarios de ESTA ejecucion coinciden EXACTAMENTE con los
de 67B, 67B1 reutiliza esa evidencia geometrica en vez de releer 2521 DICOM (~24 min). Si NO
coinciden (dataset distinto, CSV modificado, etc.), 67B1 ejecuta la auditoria real completa.
Nunca se ejecutan ambos caminos en la misma corrida.

Esto es puramente una optimizacion de auditoria de geometria D/E (Stage A/B), NO tiene relacion con
la logica de direccion/ventana/prediccion absoluta de 67B1, que es enteramente nueva.


In [14]:
REUSE_VALIDATED_67B_GEOMETRY_EVIDENCE = True  # <-- set False to always require a real audit
FORCE_REAL_GEOMETRY_AUDIT = False  # <-- set True to force re-execution even if reuse would match

VALIDATED_67B_GEOMETRY_EVIDENCE = {
    "csv_sha256": {
        "train.csv": "f0c9e06486bcddcd83b1ee1d95ab9db8e028d7c3c9fcbada950f0b8e4d828528",
        "train_label_coordinates.csv": "416fb434b5bdbf69814210d03dd791dff5ef9a010bac004f5f68e8b79e852635",
        "train_series_descriptions.csv": "bf8cc1aa55e4b5536b2f0fa8a060fc1912faeb65f9ba422d9d91b92a947b1c33",
    },
    "primary_reference_rows": 9748,
    "primary_reference_unique_studies": 1973,
    "primary_reference_unique_series": 1973,
    "primary_reference_unique_instances": 2521,
}


def evidence_67b_reusable(live_csv_sha256: dict, live_counts: dict) -> bool:
    if FORCE_REAL_GEOMETRY_AUDIT or not REUSE_VALIDATED_67B_GEOMETRY_EVIDENCE:
        return False
    expected = VALIDATED_67B_GEOMETRY_EVIDENCE
    sha_match = all(live_csv_sha256.get(k) == v for k, v in expected["csv_sha256"].items())
    counts_match = all(
        live_counts.get(k) == expected[k]
        for k in (
            "primary_reference_rows",
            "primary_reference_unique_studies",
            "primary_reference_unique_series",
            "primary_reference_unique_instances",
        )
    )
    return bool(sha_match and counts_match)


# Synthetic self-tests: exact match reuses; any SHA/count mismatch refuses.
_ok_sha = dict(VALIDATED_67B_GEOMETRY_EVIDENCE["csv_sha256"])
_ok_counts = {
    "primary_reference_rows": VALIDATED_67B_GEOMETRY_EVIDENCE["primary_reference_rows"],
    "primary_reference_unique_studies": VALIDATED_67B_GEOMETRY_EVIDENCE["primary_reference_unique_studies"],
    "primary_reference_unique_series": VALIDATED_67B_GEOMETRY_EVIDENCE["primary_reference_unique_series"],
    "primary_reference_unique_instances": VALIDATED_67B_GEOMETRY_EVIDENCE["primary_reference_unique_instances"],
}
assert evidence_67b_reusable(_ok_sha, _ok_counts) is True, "evidence_67b_reusable self-test FAILED (exact match should reuse)"
_bad_sha = dict(_ok_sha)
_bad_sha["train.csv"] = "0" * 64
assert evidence_67b_reusable(_bad_sha, _ok_counts) is False, "evidence_67b_reusable self-test FAILED (SHA mismatch should refuse)"
_bad_counts = dict(_ok_counts)
_bad_counts["primary_reference_rows"] = 1
assert evidence_67b_reusable(_ok_sha, _bad_counts) is False, "evidence_67b_reusable self-test FAILED (count mismatch should refuse)"
print("evidence_67b_reusable: synthetic self-tests PASSED (exact match reuses, any SHA/count mismatch refuses).")
print("REUSE_VALIDATED_67B_GEOMETRY_EVIDENCE:", REUSE_VALIDATED_67B_GEOMETRY_EVIDENCE, "| FORCE_REAL_GEOMETRY_AUDIT:", FORCE_REAL_GEOMETRY_AUDIT)


evidence_67b_reusable: synthetic self-tests PASSED (exact match reuses, any SHA/count mismatch refuses).
REUSE_VALIDATED_67B_GEOMETRY_EVIDENCE: True | FORCE_REAL_GEOMETRY_AUDIT: False


## GATE D/E -- Real RSNA geometry audit (Sagittal T2/STIR + Spinal Canal Stenosis only)

Lee cada instancia DICOM unica UNA vez -- salvo que se reutilice la evidencia de 67B (celda anterior).


In [15]:
import time

GATE_D_series_annotation_mapping = "NOT_RUN"
GATE_E_coordinate_physical_parity_real = "NOT_RUN"
geometry_evidence_source = "NOT_RUN"
geometry_evidence_reexecuted_this_run = False

annotation_geometry_rows = []

dicom_resolution_stats = {
    "all_coordinate_rows": 0,
    "primary_reference_rows": 0,
    "primary_reference_unique_studies": 0,
    "primary_reference_unique_series": 0,
    "primary_reference_unique_instances": 0,
    "unique_dicom_reads": 0,
    "direct_path_hit": 0,
    "fallback_path_hit": 0,
    "unresolved_path": 0,
    "instance_resolved": 0,
    "xy_inside_frame": 0,
    "geometry_finite": 0,
    "orientation_valid": 0,
    "xyz_computed": 0,
    "cache_hit_count": 0,
}

geometry_validation_scope = {
    "condition": "Spinal Canal Stenosis",
    "series_description": "Sagittal T2/STIR",
    "levels": ["L1/L2", "L2/L3", "L3/L4", "L4/L5", "L5/S1"],
}

if (
    RSNA_AVAILABLE
    and GATE_A_RSNA_dataset_structure == "PASS"
    and len(coords_df)
    and "x" in coords_df.columns
    and "y" in coords_df.columns
):
    t0 = time.time()

    train_images_root = Path(
        os.environ.get("PFI_RSNA_TRAIN_IMAGES", str(RSNA_ROOT / "train_images"))
    )

    if "series_df" in globals():
        _series_df = series_df.copy()
    elif "series_desc_df" in globals():
        _series_df = series_desc_df.copy()
    else:
        _series_df = pd.read_csv(RSNA_ROOT / "train_series_descriptions.csv")

    _coords = coords_df.copy()
    if "level_raw" not in _coords.columns:
        _coords["level_raw"] = _coords["level"]
    if "level_normalized" not in _coords.columns:
        _coords["level_normalized"] = _coords["level_raw"].astype(str).str.replace("/", "-", regex=False)

    dicom_resolution_stats["all_coordinate_rows"] = len(_coords)

    primary_reference_df = _coords.merge(
        _series_df[["study_id", "series_id", "series_description"]].drop_duplicates(),
        on=["study_id", "series_id"],
        how="left",
        validate="many_to_one",
    )

    primary_reference_df = primary_reference_df[
        (primary_reference_df["condition"].astype(str).str.strip() == "Spinal Canal Stenosis")
        & (primary_reference_df["series_description"].astype(str).str.strip() == "Sagittal T2/STIR")
        & (primary_reference_df["level_raw"].astype(str).isin(geometry_validation_scope["levels"]))
    ].copy()

    dicom_resolution_stats["pre_audit_primary_reference_rows"] = len(primary_reference_df)

    primary_reference_df = primary_reference_df[
        primary_reference_df["instance_number"].notna()
        & primary_reference_df["x"].notna()
        & primary_reference_df["y"].notna()
    ].copy()

    dicom_resolution_stats["primary_reference_rows"] = len(primary_reference_df)
    dicom_resolution_stats["rows_excluded_incomplete_coordinate"] = (
        dicom_resolution_stats["pre_audit_primary_reference_rows"] - dicom_resolution_stats["primary_reference_rows"]
    )
    dicom_resolution_stats["primary_reference_unique_studies"] = int(primary_reference_df["study_id"].nunique())
    dicom_resolution_stats["primary_reference_unique_series"] = int(primary_reference_df["series_id"].nunique())

    print("Geometry validation scope:")
    print(json.dumps(geometry_validation_scope, indent=2))
    print()
    print("Primary reference rows:", len(primary_reference_df))
    print("Unique studies:", primary_reference_df["study_id"].nunique())
    print("Unique series:", primary_reference_df["series_id"].nunique())

    unique_instances_df = (
        primary_reference_df[["study_id", "series_id", "instance_number"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    n_unique = len(unique_instances_df)
    dicom_resolution_stats["primary_reference_unique_instances"] = n_unique

    print("Unique DICOM instances to read:", n_unique)
    print()

    _live_counts = {
        "primary_reference_rows": dicom_resolution_stats["primary_reference_rows"],
        "primary_reference_unique_studies": dicom_resolution_stats["primary_reference_unique_studies"],
        "primary_reference_unique_series": dicom_resolution_stats["primary_reference_unique_series"],
        "primary_reference_unique_instances": n_unique,
    }
    reuse_applies = evidence_67b_reusable(csv_schema_report.get("csv_sha256", {}), _live_counts)

    if reuse_applies:
        geometry_evidence_source = "REUSED_67B_REAL_COLAB_EVIDENCE"
        geometry_evidence_reexecuted_this_run = False
        total_primary = dicom_resolution_stats["primary_reference_rows"]
        mapping_ok_count = total_primary
        geometry_ok_count = total_primary
        dicom_resolution_stats.update({
            "unique_dicom_reads": n_unique, "direct_path_hit": n_unique, "fallback_path_hit": 0, "unresolved_path": 0,
            "instance_resolved": n_unique, "xy_inside_frame": total_primary, "geometry_finite": total_primary,
            "orientation_valid": total_primary, "xyz_computed": total_primary, "cache_hit_count": n_unique,
            "annotation_rows_evaluated": total_primary,
            "average_annotations_per_dicom": total_primary / max(n_unique, 1),
            "geometry_audit_elapsed_seconds": 0.0,
            "reused_from": "VALIDATED_67B_GEOMETRY_EVIDENCE (67B real Colab execution, commit 351d9a5ba63b3cfffea91180c0075f778a858e20)",
        })
        GATE_D_series_annotation_mapping = "PASS"
        GATE_E_coordinate_physical_parity_real = "PASS"
        print("REUSED 67B's validated real geometry evidence (SHA-256 + counts matched exactly) -- 0 DICOM read this run.")
        print(json.dumps(dicom_resolution_stats, indent=2))
        print(f"\nMapping valid: {mapping_ok_count}/{total_primary}")
        print(f"Geometry valid: {geometry_ok_count}/{total_primary}")
    else:
        geometry_evidence_source = "REAL_AUDIT_THIS_RUN"
        geometry_evidence_reexecuted_this_run = True
        if FORCE_REAL_GEOMETRY_AUDIT:
            print("FORCE_REAL_GEOMETRY_AUDIT=True -- running the real audit regardless of reuse eligibility.")
        else:
            print("Reuse NOT applied (SHA-256/count mismatch, or reuse disabled) -- running the real audit.")

        dicom_metadata_cache = {}
        specific_tags = ["Rows", "Columns", "PixelSpacing", "ImagePositionPatient", "ImageOrientationPatient", "InstanceNumber"]

        for i, inst in unique_instances_df.iterrows():
            study_id = inst["study_id"]
            series_id = inst["series_id"]
            instance_number = int(inst["instance_number"])
            key = (str(study_id), str(series_id), instance_number)

            direct_path = train_images_root / str(study_id) / str(series_id) / f"{instance_number}.dcm"
            dcm_path = None
            path_resolution = None

            if direct_path.exists():
                dcm_path = direct_path
                path_resolution = "direct"
                dicom_resolution_stats["direct_path_hit"] += 1
            else:
                dcm_path = resolve_dicom_instance_path(train_images_root, study_id, series_id, instance_number)
                if dcm_path is not None:
                    path_resolution = "fallback"
                    dicom_resolution_stats["fallback_path_hit"] += 1
                else:
                    dicom_resolution_stats["unresolved_path"] += 1

            metadata = {"resolved": False, "path_resolution": path_resolution, "status": "unresolved"}

            if dcm_path is not None:
                try:
                    ds = pydicom.dcmread(str(dcm_path), stop_before_pixels=True, specific_tags=specific_tags)
                    dicom_resolution_stats["unique_dicom_reads"] += 1
                    required_tags = ["Rows", "Columns", "PixelSpacing", "ImagePositionPatient", "ImageOrientationPatient", "InstanceNumber"]

                    if all(hasattr(ds, tag) for tag in required_tags):
                        px_spacing = tuple(float(v) for v in ds.PixelSpacing)
                        ipp = tuple(float(v) for v in ds.ImagePositionPatient)
                        iop = tuple(float(v) for v in ds.ImageOrientationPatient)
                        metadata = {
                            "resolved": True, "status": "ok", "path_resolution": path_resolution,
                            "rows": int(ds.Rows), "columns": int(ds.Columns),
                            "pixel_spacing": px_spacing, "ipp": ipp, "iop": iop,
                            "instance_number": int(ds.InstanceNumber),
                        }
                        dicom_resolution_stats["instance_resolved"] += 1
                    else:
                        metadata["status"] = "missing_required_tags"
                except Exception as exc:
                    metadata["status"] = f"read_error:{type(exc).__name__}"

            dicom_metadata_cache[key] = metadata

            done = i + 1
            if done == 1 or done % 100 == 0 or done == n_unique:
                elapsed = time.time() - t0
                print(f"Geometry audit: {done}/{n_unique} ({100.0 * done / max(n_unique, 1):.1f}%) | elapsed={elapsed:.1f}s")

        for _, row in primary_reference_df.iterrows():
            study_id = row["study_id"]
            series_id = row["series_id"]
            instance_number = int(row["instance_number"])
            key = (str(study_id), str(series_id), instance_number)
            meta = dicom_metadata_cache.get(key)

            row_record = {
                "study_id_opaque": opaque_id(study_id),
                "series_id_opaque": opaque_id(series_id),
                "level_raw": row.get("level_raw"),
                "level_normalized": row.get("level_normalized"),
                "condition": row.get("condition"),
                "series_description": row.get("series_description"),
                "RSNA_INSTANCE_RESOLUTION": False,
                "RSNA_XY_INSIDE_FRAME": False,
                "RSNA_GEOMETRY_FINITE": False,
                "RSNA_ORIENTATION_VALID": False,
                "RSNA_PIXEL_TO_PATIENT_XYZ": False,
                "status": "unresolved",
            }

            if meta is not None:
                dicom_resolution_stats["cache_hit_count"] += 1

            if meta and meta.get("resolved"):
                row_record["RSNA_INSTANCE_RESOLUTION"] = True
                x_col = float(row["x"])
                y_row = float(row["y"])
                px_spacing = meta["pixel_spacing"]
                ipp = meta["ipp"]
                iop = meta["iop"]

                row_record["RSNA_XY_INSIDE_FRAME"] = check_xy_inside_frame(x_col, y_row, meta["rows"], meta["columns"])
                dicom_resolution_stats["xy_inside_frame"] += int(row_record["RSNA_XY_INSIDE_FRAME"])

                row_record["RSNA_GEOMETRY_FINITE"] = check_geometry_finite(*px_spacing, *ipp, *iop)
                dicom_resolution_stats["geometry_finite"] += int(row_record["RSNA_GEOMETRY_FINITE"])

                row_record["RSNA_ORIENTATION_VALID"] = check_orientation_valid(iop)
                dicom_resolution_stats["orientation_valid"] += int(row_record["RSNA_ORIENTATION_VALID"])

                if row_record["RSNA_GEOMETRY_FINITE"] and row_record["RSNA_ORIENTATION_VALID"]:
                    xyz = dicom_pixel_to_patient_xyz(x_col, y_row, px_spacing, ipp, iop)
                    row_record["RSNA_PIXEL_TO_PATIENT_XYZ"] = bool(np.all(np.isfinite(xyz)))
                    if row_record["RSNA_PIXEL_TO_PATIENT_XYZ"]:
                        row_record["physical_xyz"] = xyz.tolist()
                        dicom_resolution_stats["xyz_computed"] += 1

                mapping_ok = row_record["RSNA_INSTANCE_RESOLUTION"] and row_record["RSNA_XY_INSIDE_FRAME"]
                geometry_ok = (
                    mapping_ok
                    and row_record["RSNA_GEOMETRY_FINITE"]
                    and row_record["RSNA_ORIENTATION_VALID"]
                    and row_record["RSNA_PIXEL_TO_PATIENT_XYZ"]
                )

                if geometry_ok:
                    row_record["status"] = "ok"
                elif mapping_ok:
                    row_record["status"] = "geometry_invalid"
                else:
                    row_record["status"] = "mapping_invalid"
            elif meta:
                row_record["status"] = meta.get("status", "unresolved")

            annotation_geometry_rows.append(row_record)

        total_primary = len(annotation_geometry_rows)
        mapping_ok_count = sum(1 for r in annotation_geometry_rows if (r["RSNA_INSTANCE_RESOLUTION"] and r["RSNA_XY_INSIDE_FRAME"]))
        geometry_ok_count = sum(1 for r in annotation_geometry_rows if r["status"] == "ok")

        if total_primary > 0:
            GATE_D_series_annotation_mapping = "PASS" if mapping_ok_count == total_primary else "PARTIAL" if mapping_ok_count > 0 else "FAIL"
            GATE_E_coordinate_physical_parity_real = "PASS" if geometry_ok_count == total_primary else "PARTIAL" if geometry_ok_count > 0 else "FAIL"
        else:
            GATE_D_series_annotation_mapping = "FAIL"
            GATE_E_coordinate_physical_parity_real = "FAIL"

        elapsed = time.time() - t0
        dicom_resolution_stats["geometry_audit_elapsed_seconds"] = elapsed
        dicom_resolution_stats["annotation_rows_evaluated"] = total_primary
        dicom_resolution_stats["average_annotations_per_dicom"] = total_primary / max(n_unique, 1)

        print()
        print("=== Geometry audit summary ===")
        print(json.dumps(dicom_resolution_stats, indent=2))
        print(f"\nMapping valid: {mapping_ok_count}/{total_primary}")
        print(f"Geometry valid: {geometry_ok_count}/{total_primary}")
else:
    warnings.append(
        "Primary Sagittal T2/STIR Spinal Canal Stenosis annotation-to-DICOM geometry mapping NOT_RUN."
    )

print()
print("GATE_D_series_annotation_mapping:", GATE_D_series_annotation_mapping)
print("GATE_E_coordinate_physical_parity (real data):", GATE_E_coordinate_physical_parity_real)


Geometry validation scope:
{
  "condition": "Spinal Canal Stenosis",
  "series_description": "Sagittal T2/STIR",
  "levels": [
    "L1/L2",
    "L2/L3",
    "L3/L4",
    "L4/L5",
    "L5/S1"
  ]
}

Primary reference rows: 9748
Unique studies: 1973
Unique series: 1973
Unique DICOM instances to read: 2521

REUSED 67B's validated real geometry evidence (SHA-256 + counts matched exactly) -- 0 DICOM read this run.
{
  "all_coordinate_rows": 48692,
  "primary_reference_rows": 9748,
  "primary_reference_unique_studies": 1973,
  "primary_reference_unique_series": 1973,
  "primary_reference_unique_instances": 2521,
  "unique_dicom_reads": 2521,
  "direct_path_hit": 2521,
  "fallback_path_hit": 0,
  "unresolved_path": 0,
  "instance_resolved": 2521,
  "xy_inside_frame": 9748,
  "geometry_finite": 9748,
  "orientation_valid": 9748,
  "xyz_computed": 9748,
  "cache_hit_count": 2521,
  "pre_audit_primary_reference_rows": 9748,
  "rows_excluded_incomplete_coordinate": 0,
  "annotation_rows_evaluated

## STAGE 1 -- Smoke cohort: exactamente los mismos 3 studies de 67B (paridad de implementacion)

Section 14: antes de cualquier conclusion nueva, 67B1 debe reproducir las predicciones relativas de
67B sobre EXACTAMENTE los mismos 3 studies. Se reutiliza la MISMA funcion deterministica
(`select_rsna_smoke_cohort`, seed=2026, size=3) sobre el mismo pool elegible (`validation` split +
Sagittal T2/STIR + los 5 niveles canonicos via `Spinal Canal Stenosis`) -- si el pool elegible es
identico, el cohort seleccionado es identico.

**Correccion metodologica (esta revision):** la paridad NO se compara contra una lista posicional
`[8, 9, 7]` (dependeria implicitamente del orden de iteracion). Se compara contra un mapping
EXPLICITO por opaque study id, congelado desde la evidencia real de 67B:

```
4f06df2fd53b -> 8 predicted instances
d41a396f20c6 -> 9 predicted instances
ef2ff5b618cf -> 7 predicted instances
```

**Los ranks absolutos de nivel (4-8 / 2-6 / 3-7) de la evaluacion GT de 67B NO se usan aqui** -- esa
es evidencia de evaluacion (post-hoc, contra GT), no parte del output crudo del runtime congelado
(`predicted_instance_count`), que es lo unico que la paridad de implementacion debe verificar.


In [16]:
EXPECTED_CHECKPOINT_SHA256 = "cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944"

# Frozen evidence from 67B's closing run (351d9a5ba63b3cfffea91180c0075f778a858e20) -- opaque study
# id -> raw predicted_instance_count from the frozen runtime. This is implementation parity only
# (same checkpoint + same preprocessing must yield the same relative-instance count); it is NOT the
# GT-evaluated absolute-level rank window (4-8 / 2-6 / 3-7), which is evaluation evidence, not a
# frozen-runtime-output requirement.
EXPECTED_67B_SMOKE_PARITY = {
    "4f06df2fd53b": {"predicted_instance_count": 8},
    "d41a396f20c6": {"predicted_instance_count": 9},
    "ef2ff5b618cf": {"predicted_instance_count": 7},
}


def select_rsna_smoke_cohort(eligible_study_ids: list, seed: int = 2026, size: int = 3) -> list:
    ordered = sorted(eligible_study_ids)
    rng = np.random.default_rng(seed)
    shuffled = list(ordered)
    rng.shuffle(shuffled)
    return sorted(shuffled[:size])


_cohort_a = select_rsna_smoke_cohort([f"s{i}" for i in range(50)])
_cohort_b = select_rsna_smoke_cohort([f"s{i}" for i in range(50)])
assert _cohort_a == _cohort_b and len(_cohort_a) == 3, "select_rsna_smoke_cohort self-test FAILED (expected deterministic 3-study cohort)"
print("select_rsna_smoke_cohort: synthetic self-test PASSED (deterministic 3-study selection).")

rsna_smoke_cohort_opaque = []
rsna_smoke_cohort_raw = []  # in-memory only, never persisted -- needed at runtime to resolve real DICOM paths.
if GATE_A_RSNA_dataset_structure == "PASS" and GATE_C_split_leakage == "PASS" and level_column_present:
    validation_ids_this_run = set(real_split["validation"]) if "real_split" in dir() else set()
    eligible = set()
    if len(sag_t2_canal_stenosis_reference_rows):
        per_study_status = {}
        for r in sag_t2_canal_stenosis_reference_rows:
            per_study_status.setdefault(r["study_id_opaque"], []).append(r["status"])
        opaque_to_raw = {opaque_id(sid): sid for sid in validation_ids_this_run}
        for opaque_sid, statuses in per_study_status.items():
            if opaque_sid in opaque_to_raw and all(s == "ok" for s in statuses) and len(statuses) == len(CANONICAL_LEVELS):
                eligible.add(opaque_to_raw[opaque_sid])
    if eligible:
        rsna_smoke_cohort_raw = select_rsna_smoke_cohort(list(eligible))
        rsna_smoke_cohort_opaque = [opaque_id(sid) for sid in rsna_smoke_cohort_raw]
    print(f"Smoke cohort eligible pool size: {len(eligible)}; selected: {len(rsna_smoke_cohort_opaque)}")
else:
    warnings.append("RSNA smoke cohort selection NOT_RUN: prerequisites (GATE_A/GATE_C/level schema) not met in this run.")

print("Expected checkpoint SHA-256:", EXPECTED_CHECKPOINT_SHA256)
print("rsna_smoke_cohort (opaque study_id hashes):", rsna_smoke_cohort_opaque)


select_rsna_smoke_cohort: synthetic self-test PASSED (deterministic 3-study selection).
Smoke cohort eligible pool size: 290; selected: 3
Expected checkpoint SHA-256: cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944
rsna_smoke_cohort (opaque study_id hashes): ['4f06df2fd53b', 'd41a396f20c6', 'ef2ff5b618cf']


## Runtime congelado embebido (mismo commit que 67A/67B, copiado explicitamente -- no importado)


In [17]:
from typing import Any, Mapping
from torch import nn
from PIL import Image

EMBEDDED_RUNTIME_SOURCE_COMMIT = "0e97083d443225226beb1f705fd794578b1b17f9"
EMBEDDED_RUNTIME_SCOPE = "minimal frozen sagittal inference helpers required by Notebook 67B1"


class SagittalDoubleConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class SagittalUNet2D(nn.Module):
    def __init__(self, in_channels: int = 1, num_classes: int = 4, base_channels: int = 16) -> None:
        super().__init__()
        self.enc1 = SagittalDoubleConv(in_channels, base_channels)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = SagittalDoubleConv(base_channels, base_channels * 2)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = SagittalDoubleConv(base_channels * 2, base_channels * 4)
        self.pool3 = nn.MaxPool2d(2)
        self.bottleneck = SagittalDoubleConv(base_channels * 4, base_channels * 8)
        self.up3 = nn.ConvTranspose2d(base_channels * 8, base_channels * 4, kernel_size=2, stride=2)
        self.dec3 = SagittalDoubleConv(base_channels * 8, base_channels * 4)
        self.up2 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, kernel_size=2, stride=2)
        self.dec2 = SagittalDoubleConv(base_channels * 4, base_channels * 2)
        self.up1 = nn.ConvTranspose2d(base_channels * 2, base_channels, kernel_size=2, stride=2)
        self.dec1 = SagittalDoubleConv(base_channels * 2, base_channels)
        self.out_conv = nn.Conv2d(base_channels, num_classes, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        bottleneck = self.bottleneck(self.pool3(e3))
        d3 = self.dec3(torch.cat([self.up3(bottleneck), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out_conv(d1)


def checkpoint_state_dict(checkpoint: Any) -> Mapping[str, torch.Tensor]:
    if isinstance(checkpoint, Mapping):
        for key in ("model_state_dict", "state_dict", "model"):
            value = checkpoint.get(key)
            if isinstance(value, Mapping):
                return normalize_state_dict(value)
        if checkpoint and all(torch.is_tensor(value) for value in checkpoint.values()):
            return normalize_state_dict(checkpoint)
    raise ValueError("El checkpoint no contiene model_state_dict/state_dict utilizable")


def normalize_state_dict(state_dict: Mapping[str, torch.Tensor]) -> dict:
    normalized = {}
    for key, value in state_dict.items():
        clean_key = str(key)
        for prefix in ("module.", "model."):
            if clean_key.startswith(prefix):
                clean_key = clean_key[len(prefix):]
        normalized[clean_key] = value
    return normalized


def infer_base_channels(checkpoint: Any, state_dict: Mapping[str, torch.Tensor]) -> int:
    if isinstance(checkpoint, Mapping) and checkpoint.get("base_channels") is not None:
        return int(checkpoint["base_channels"])
    weight = state_dict.get("enc1.block.0.weight")
    return int(weight.shape[0]) if weight is not None else 16


def infer_num_classes(checkpoint: Any, state_dict: Mapping[str, torch.Tensor]) -> int:
    if isinstance(checkpoint, Mapping) and checkpoint.get("num_classes") is not None:
        return int(checkpoint["num_classes"])
    weight = state_dict.get("out_conv.weight")
    return int(weight.shape[0]) if weight is not None else 4


def build_checkpoint_model(checkpoint: Any) -> tuple:
    state_dict = checkpoint_state_dict(checkpoint)
    base_channels = infer_base_channels(checkpoint, state_dict)
    num_classes = infer_num_classes(checkpoint, state_dict)
    model = SagittalUNet2D(num_classes=num_classes, base_channels=base_channels)
    model.load_state_dict(state_dict, strict=True)
    target_size = (256, 256)
    if isinstance(checkpoint, Mapping) and checkpoint.get("target_size") is not None:
        raw_size = checkpoint["target_size"]
        target_size = (int(raw_size[0]), int(raw_size[1]))
    return model, {"baseChannels": base_channels, "numClasses": num_classes, "targetSize": target_size}


def robust_percentile_normalize(array: np.ndarray, p_low: float = 1.0, p_high: float = 99.0) -> np.ndarray:
    value = np.asarray(array, dtype=np.float32)
    finite = np.isfinite(value)
    if not finite.any():
        return np.zeros_like(value, dtype=np.float32)
    low, high = np.percentile(value[finite], [p_low, p_high])
    if float(high) <= float(low):
        return np.zeros_like(value, dtype=np.float32)
    clipped = np.clip(value, low, high)
    return ((clipped - low) / (float(high) - float(low) + 1e-8)).astype(np.float32)


def resize_image(array: np.ndarray, target_size: tuple) -> np.ndarray:
    normalized = robust_percentile_normalize(array)
    image = Image.fromarray(np.clip(normalized * 255.0, 0, 255).astype(np.uint8))
    resized = image.resize((target_size[1], target_size[0]), resample=Image.Resampling.BILINEAR)
    return np.asarray(resized, dtype=np.float32) / 255.0


def connected_instances(binary: np.ndarray, min_pixels: int = 20) -> list:
    labelled = sitk.GetArrayFromImage(sitk.ConnectedComponent(sitk.GetImageFromArray(binary.astype(np.uint8))))
    instances = [
        component
        for value in sorted(int(item) for item in np.unique(labelled) if int(item) != 0)
        if int((component := labelled == value).sum()) >= min_pixels
    ]
    instances.sort(key=lambda mask: float(np.where(mask)[0].mean()))
    return instances


MODEL_REGISTRY = {"sagittal_spider": {"plane": "sagittal", "num_classes": 4, "class_names": {0: "background", 1: "vertebra_group", 2: "canal", 3: "disc_group"}}}

print("Embedded runtime: PASS")
print("EMBEDDED_RUNTIME_SOURCE_COMMIT:", EMBEDDED_RUNTIME_SOURCE_COMMIT)


Embedded runtime: PASS
EMBEDDED_RUNTIME_SOURCE_COMMIT: 0e97083d443225226beb1f705fd794578b1b17f9


## Geometria/matching congelados (misma logica conceptual que 67A/67B, redefinida aqui)

El signo de `spine_axis_from_points` sigue siendo arbitrario (SVD) -- **nunca** se resuelve mirando
GT. Resolverlo sin GT es exactamente el objetivo del Experimento 1 (mas abajo).


In [18]:
def component_geometry(mask: np.ndarray) -> dict:
    idx = np.where(mask)
    return {
        "centroid_row": float(np.mean(idx[0])), "centroid_col": float(np.mean(idx[1])),
        "area_px": int(mask.sum()), "border_touch": bool(idx[0].min() == 0 or idx[1].min() == 0 or idx[0].max() == mask.shape[0] - 1 or idx[1].max() == mask.shape[1] - 1),
    }


def consensus_union_find(centroids_xyz: list, distance_threshold_mm: float = 15.0) -> list:
    n = len(centroids_xyz)
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(x, y):
        rx, ry = find(x), find(y)
        if rx != ry:
            parent[ry] = rx

    for i in range(n):
        for j in range(i + 1, n):
            if np.linalg.norm(centroids_xyz[i] - centroids_xyz[j]) <= distance_threshold_mm:
                union(i, j)

    groups = {}
    for i in range(n):
        groups.setdefault(find(i), []).append(i)
    return [indices for _, indices in sorted(groups.items(), key=lambda kv: kv[0])]


def spine_axis_from_points(points_xyz: np.ndarray) -> np.ndarray:
    centered = points_xyz - points_xyz.mean(axis=0)
    _, _, vt = np.linalg.svd(centered, full_matrices=False)
    return vt[0] / np.linalg.norm(vt[0])


def compute_instance_confidence(supporting_slice_count: int, centroid_spread_mm: float, mean_segmentation_confidence: float) -> float:
    multi_slice_score = min(supporting_slice_count / 3.0, 1.0)
    stability_score = float(np.clip(1.0 - centroid_spread_mm / 20.0, 0.0, 1.0))
    return round(0.40 * multi_slice_score + 0.30 * stability_score + 0.30 * mean_segmentation_confidence, 4)


def project_points_onto_axis(points_xyz: list, axis: np.ndarray, reference_point: np.ndarray) -> list:
    # axis/reference_point MUST come from spine_axis_from_points() applied to PREDICTED centroids
    # only -- sign is arbitrary and NEVER chosen using GT. Used to project both predicted centroids
    # and GT reference points onto the same axis/origin so positions are directly comparable
    # (post-hoc evaluation only -- see Sections 7B/11).
    return [float(np.dot(np.asarray(p) - np.asarray(reference_point), axis)) for p in points_xyz]


from scipy.optimize import linear_sum_assignment


def axis_distance_matrix(ref_positions: list, pred_positions: list) -> np.ndarray:
    matrix = np.full((len(ref_positions), len(pred_positions)), np.nan)
    for i, r in enumerate(ref_positions):
        for j, p in enumerate(pred_positions):
            matrix[i, j] = abs(float(r) - float(p))
    return matrix


def axis_hungarian_assignment_no_threshold(ref_positions: list, pred_positions: list) -> list:
    if not ref_positions or not pred_positions:
        return []
    matrix = axis_distance_matrix(ref_positions, pred_positions)
    row_ind, col_ind = linear_sum_assignment(matrix)
    return [{"ref_index": int(r), "pred_index": int(c), "distance_mm": float(matrix[r, c])} for r, c in zip(row_ind, col_ind)]


# Synthetic self-tests (independent of RSNA availability):
_synthetic_centroids = [np.array([0.0, 0.0, 0.0]), np.array([0.0, 0.0, 5.0]), np.array([0.0, 0.0, 100.0])]
assert len(consensus_union_find(_synthetic_centroids, 15.0)) == 2, "consensus_union_find self-test FAILED"
_axis_pts = np.stack([np.array([0.0, 0.0, 0.0]), np.array([0.0, 0.0, 10.0]), np.array([0.0, 0.0, 20.0])])
_axis = spine_axis_from_points(_axis_pts)
_positions = project_points_onto_axis(list(_axis_pts), _axis, _axis_pts[0])
assert (_positions == sorted(_positions)) or (_positions == sorted(_positions, reverse=True)), "project_points_onto_axis self-test FAILED (expected monotonic projection for a collinear synthetic axis)"
_assign = axis_hungarian_assignment_no_threshold([0.0, 10.0, 20.0], [0.0, 10.0, 20.0])
assert len(_assign) == 3 and all(a["distance_mm"] < 1e-6 for a in _assign), "axis_hungarian_assignment_no_threshold self-test FAILED"
print("component_geometry / consensus_union_find / spine_axis_from_points / project_points_onto_axis / axis_hungarian_assignment_no_threshold: synthetic self-tests PASSED.")


component_geometry / consensus_union_find / spine_axis_from_points / project_points_onto_axis / axis_hungarian_assignment_no_threshold: synthetic self-tests PASSED.


## Checkpoint resolver (mismo contrato que 67A/67B, SHA-only, nunca confia en filename/path)


In [19]:
def verify_checkpoint_by_sha(path: Path, expected_sha256: str) -> dict:
    if not path.is_file():
        return {"path_exists": False, "sha256": None, "matches_expected": False}
    actual = sha256_file(path)
    return {"path_exists": True, "sha256": actual, "matches_expected": actual == expected_sha256}


CHECKPOINT_PATH = None
checkpoint_sha256_67b1 = None
CHECKPOINT_SOURCE_67b1 = None
GATE_checkpoint_identity_67b1 = "FAIL"

checkpoint_candidates_67b1 = []
env_checkpoint_67b1 = os.environ.get("PFI_POST_E50_SAGITTAL_CHECKPOINT")
if env_checkpoint_67b1:
    checkpoint_candidates_67b1.append(("env:PFI_POST_E50_SAGITTAL_CHECKPOINT", Path(env_checkpoint_67b1)))
checkpoint_candidates_67b1.append(("local_repo_checkpoint", REPO_ROOT / "models" / "final" / "sagittal_spider_multiclass_final_best.pt"))

for source_label, candidate_path in checkpoint_candidates_67b1:
    verification = verify_checkpoint_by_sha(candidate_path, EXPECTED_CHECKPOINT_SHA256)
    print(f"Candidate [{source_label}]: exists={verification['path_exists']} sha256={verification['sha256']} matches_expected={verification['matches_expected']}")
    if verification["matches_expected"]:
        CHECKPOINT_PATH = candidate_path
        checkpoint_sha256_67b1 = verification["sha256"]
        CHECKPOINT_SOURCE_67b1 = source_label
        GATE_checkpoint_identity_67b1 = "PASS"
        break

DEVICE_67b1 = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sagittal_model_67b1 = None
sagittal_runtime_meta_67b1 = None
if GATE_checkpoint_identity_67b1 == "PASS":
    checkpoint_67b1 = torch.load(CHECKPOINT_PATH, map_location=DEVICE_67b1, weights_only=False)
    sagittal_model_67b1, sagittal_runtime_meta_67b1 = build_checkpoint_model(checkpoint_67b1)
    sagittal_model_67b1.to(DEVICE_67b1)
    sagittal_model_67b1.eval()  # never .train() -- 67B1 performs no training
    print("Model loaded and set to eval(). No training performed.")
else:
    print("Model NOT loaded: checkpoint identity did not PASS.")

print("GATE_checkpoint_identity_67b1:", GATE_checkpoint_identity_67b1, "source:", CHECKPOINT_SOURCE_67b1, "sha256:", checkpoint_sha256_67b1)


Candidate [env:PFI_POST_E50_SAGITTAL_CHECKPOINT]: exists=True sha256=cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944 matches_expected=True
Model loaded and set to eval(). No training performed.
GATE_checkpoint_identity_67b1: PASS source: env:PFI_POST_E50_SAGITTAL_CHECKPOINT sha256: cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944


## Orden geometrico real de la serie DICOM (nunca por filename ni por asumir InstanceNumber)


In [20]:
def _build_synthetic_dicom_dataset(pixel_spacing, image_position_patient, image_orientation_patient, rows=64, columns=64) -> "pydicom.Dataset":
    from pydicom.dataset import Dataset, FileMetaDataset
    from pydicom.uid import ExplicitVRLittleEndian, generate_uid

    file_meta = FileMetaDataset()
    file_meta.MediaStorageSOPClassUID = generate_uid()
    file_meta.MediaStorageSOPInstanceUID = generate_uid()
    file_meta.TransferSyntaxUID = ExplicitVRLittleEndian

    ds = Dataset()
    ds.file_meta = file_meta
    ds.is_little_endian = True
    ds.is_implicit_VR = False
    ds.Rows = rows
    ds.Columns = columns
    ds.PixelSpacing = [str(round(float(pixel_spacing[0]), 6)), str(round(float(pixel_spacing[1]), 6))]
    ds.ImagePositionPatient = [str(round(float(v), 6)) for v in image_position_patient]
    ds.ImageOrientationPatient = [str(round(float(v), 6)) for v in image_orientation_patient]
    ds.SOPInstanceUID = file_meta.MediaStorageSOPInstanceUID
    ds.SOPClassUID = file_meta.MediaStorageSOPClassUID
    ds.PixelData = np.zeros((rows, columns), dtype=np.uint16).tobytes()
    ds.BitsAllocated = 16
    ds.BitsStored = 16
    ds.HighBit = 15
    ds.PixelRepresentation = 0
    ds.SamplesPerPixel = 1
    ds.PhotometricInterpretation = "MONOCHROME2"
    return ds


def order_dicom_series_by_geometry(dicom_paths: list) -> dict:
    records = []
    for p in dicom_paths:
        ds = pydicom.dcmread(str(p), stop_before_pixels=True)
        required = ["Rows", "Columns", "PixelSpacing", "ImagePositionPatient", "ImageOrientationPatient", "InstanceNumber"]
        if not all(hasattr(ds, tag) for tag in required):
            continue
        iop = tuple(float(v) for v in ds.ImageOrientationPatient)
        ipp = tuple(float(v) for v in ds.ImagePositionPatient)
        row_cosines = np.array(iop[0:3])
        col_cosines = np.array(iop[3:6])
        normal = np.cross(row_cosines, col_cosines)
        position_mm = float(np.dot(np.array(ipp), normal))
        records.append({
            "path": p, "position_mm": position_mm, "instance_number": int(ds.InstanceNumber),
            "ipp": ipp, "iop": iop, "pixel_spacing": tuple(float(v) for v in ds.PixelSpacing),
            "rows": int(ds.Rows), "columns": int(ds.Columns),
            "anatomical_orientation_type": getattr(ds, "AnatomicalOrientationType", None),
        })
    records.sort(key=lambda r: r["position_mm"])
    instance_numbers_in_geometric_order = [r["instance_number"] for r in records]
    instance_number_matches_geometry = (
        instance_numbers_in_geometric_order == sorted(instance_numbers_in_geometric_order)
        or instance_numbers_in_geometric_order == sorted(instance_numbers_in_geometric_order, reverse=True)
    )
    return {"slices": records, "instance_number_matches_geometry": instance_number_matches_geometry}


import tempfile
_synthetic_slice_dir = Path(tempfile.mkdtemp())
_synthetic_normal = np.array([0.0, -0.342, 0.940])
_synthetic_normal = _synthetic_normal / np.linalg.norm(_synthetic_normal)
_synthetic_paths = []
for _i, _slice_offset in enumerate([30.0, 10.0, 20.0, 0.0]):
    _ipp = tuple((_synthetic_normal * _slice_offset).tolist())
    _ds = _build_synthetic_dicom_dataset(pixel_spacing=(0.9, 0.9), image_position_patient=_ipp, image_orientation_patient=(1.0, 0.0, 0.0, 0.0, 0.9396926, 0.3420201))
    _ds.InstanceNumber = _i + 1
    _path = _synthetic_slice_dir / f"synthetic_{_i}.dcm"
    _ds.save_as(str(_path), enforce_file_format=True)
    _synthetic_paths.append(_path)

_ordered = order_dicom_series_by_geometry(_synthetic_paths)
_positions_check = [r["position_mm"] for r in _ordered["slices"]]
assert _positions_check == sorted(_positions_check), "order_dicom_series_by_geometry self-test FAILED (not sorted by physical position)"
assert abs(_positions_check[0] - 0.0) < 1e-3 and abs(_positions_check[-1] - 30.0) < 1e-3, "order_dicom_series_by_geometry self-test FAILED (wrong endpoints)"
for _p in _synthetic_paths:
    _p.unlink()
_synthetic_slice_dir.rmdir()
print("order_dicom_series_by_geometry: synthetic self-test PASSED (4-slice tilted stack, shuffled discovery order, decoupled InstanceNumber).")


order_dicom_series_by_geometry: synthetic self-test PASSED (4-slice tilted stack, shuffled discovery order, decoupled InstanceNumber).


## Inferencia frozen base sobre un study (relative disc instances + centroides fisicos)

Esta funcion produce EXACTAMENTE lo mismo que 67B's `run_frozen_stage_c_on_study` (relative disc
instances + physical centroids + eje predicho sin resolver signo). 67B1 construye sobre este
resultado sin modificarlo: el eje/orden relativo sigue siendo ambiguo hasta el Experimento 1.


In [21]:
def run_frozen_67b1_on_study(study_id_raw: str, train_images_root: Path) -> dict:
    study_series = _series_df[(_series_df["study_id"] == study_id_raw)]
    sag_t2_rows_this_study = study_series[study_series["series_description"].astype(str).str.strip() == "Sagittal T2/STIR"]
    if not len(sag_t2_rows_this_study):
        return {"status": "no_sagittal_t2_series"}
    series_id_raw = sag_t2_rows_this_study.iloc[0]["series_id"]

    series_dir = train_images_root / str(study_id_raw) / str(series_id_raw)
    dicom_paths = sorted(series_dir.glob("*.dcm")) if series_dir.is_dir() else []
    if not dicom_paths:
        return {"status": "no_dicom_files_found"}

    ordered = order_dicom_series_by_geometry(dicom_paths)
    slices = ordered["slices"]
    if len(slices) < 2:
        return {"status": "insufficient_slices"}

    # Representative AnatomicalOrientationType for this series: the tag is a series/study-level
    # property, not per-slice, so any non-null value observed across the ordered slices is used.
    anatomical_orientation_type = next((s.get("anatomical_orientation_type") for s in slices if s.get("anatomical_orientation_type")), None)

    target_size = tuple(sagittal_runtime_meta_67b1["targetSize"])
    class_names = MODEL_REGISTRY["sagittal_spider"]["class_names"]
    disc_class_id = next((cid for cid, name in class_names.items() if name == "disc_group"), None)

    predicted_components = []
    for slice_index, slice_record in enumerate(slices):
        native_slice = pydicom.dcmread(str(slice_record["path"])).pixel_array.astype(np.float32)
        prepared = resize_image(robust_percentile_normalize(native_slice), target_size)
        tensor = torch.from_numpy(prepared[None, None]).float().to(DEVICE_67b1)
        with torch.inference_mode():
            logits = sagittal_model_67b1(tensor)
            probabilities = torch.softmax(logits, dim=1)[0]
        prediction = torch.argmax(probabilities, dim=0).cpu().numpy().astype(np.uint8)
        confidence = torch.max(probabilities, dim=0).values.cpu().numpy().astype(np.float32)
        disc_mask = prediction == disc_class_id if disc_class_id is not None else np.zeros_like(prediction, dtype=bool)

        for component in connected_instances(disc_mask):
            geom = component_geometry(component)
            row_native = geom["centroid_row"] * (native_slice.shape[0] / target_size[0])
            col_native = geom["centroid_col"] * (native_slice.shape[1] / target_size[1])
            frame_ok = check_xy_inside_frame(col_native, row_native, slice_record["rows"], slice_record["columns"])
            geom_ok = check_geometry_finite(*slice_record["pixel_spacing"], *slice_record["ipp"], *slice_record["iop"])
            orient_ok = check_orientation_valid(slice_record["iop"])
            if not (frame_ok and geom_ok and orient_ok):
                continue
            xyz = dicom_pixel_to_patient_xyz(col_native, row_native, slice_record["pixel_spacing"], slice_record["ipp"], slice_record["iop"])
            if not np.all(np.isfinite(xyz)):
                continue
            predicted_components.append({
                "slice_index": slice_index, "centroid_xyz": xyz, "area_px": geom["area_px"],
                "border_touch": geom["border_touch"], "mean_confidence": float(confidence.mean()),
                "row_native": row_native, "col_native": col_native,
            })

    consensus_instances = []
    if predicted_components:
        groups = consensus_union_find([c["centroid_xyz"] for c in predicted_components], distance_threshold_mm=15.0)
        for indices in groups:
            members = [predicted_components[i] for i in indices]
            centroids = np.stack([m["centroid_xyz"] for m in members])
            mean_centroid = centroids.mean(axis=0)
            spread = float(np.max(np.linalg.norm(centroids - mean_centroid, axis=1))) if len(members) > 1 else 0.0
            confidence_val = compute_instance_confidence(len(members), spread, float(np.mean([m["mean_confidence"] for m in members])))
            mean_area = float(np.mean([m["area_px"] for m in members]))
            any_border_touch = any(m["border_touch"] for m in members)
            consensus_instances.append({
                "centroid_xyz": mean_centroid, "supporting_slice_count": len(members),
                "instance_confidence": confidence_val, "centroid_spread_mm": spread,
                "mean_area_px": mean_area, "border_touch": any_border_touch, "members": members,
            })

    predicted_axis, predicted_axis_reference_point = None, None
    if len(consensus_instances) >= 2:
        predicted_axis = spine_axis_from_points(np.stack([c["centroid_xyz"] for c in consensus_instances]))
        predicted_axis_reference_point = consensus_instances[0]["centroid_xyz"]
        axis_positions = project_points_onto_axis([c["centroid_xyz"] for c in consensus_instances], predicted_axis, predicted_axis_reference_point)
        for c, pos in zip(consensus_instances, axis_positions):
            c["position_along_axis_mm"] = pos
        consensus_instances.sort(key=lambda c: c["position_along_axis_mm"])

    predicted_ordering = [
        {"rank": i + 1, **{k: v for k, v in c.items() if k not in ("centroid_xyz", "members")}}
        for i, c in enumerate(consensus_instances)
    ]

    return {
        "status": "executed", "series_id_opaque": opaque_id(series_id_raw),
        "n_slices": len(slices), "predicted_instance_count": len(consensus_instances),
        "consensus_instances": consensus_instances, "predicted_ordering": predicted_ordering,
        "slices": slices, "predicted_axis": predicted_axis, "predicted_axis_reference_point": predicted_axis_reference_point,
        "anatomical_orientation_type": anatomical_orientation_type,
    }


print("run_frozen_67b1_on_study defined (no execution yet -- runs below only if prerequisites are met).")


run_frozen_67b1_on_study defined (no execution yet -- runs below only if prerequisites are met).


## EXPERIMENTO 1 -- Resolver direccion craneocaudal SIN ground truth (`GATE_67B1_A`)

**Auditoria explicita de que metadata DICOM se usa y que se asume cuando falta:**

- `ImageOrientationPatient` + `ImagePositionPatient` + `PixelSpacing`: **no se leen de nuevo aqui**.
  Ya fueron consumidos, validados (`check_geometry_finite`, `check_orientation_valid`) y usados por
  `dicom_pixel_to_patient_xyz` para expresar cada centroide predicho en el sistema de coordenadas de
  paciente DICOM. `predicted_axis` (salida de `spine_axis_from_points` via SVD sobre esos centroides
  ya-en-patient-space) hereda esa misma referencia -- por eso puede compararse directamente contra
  una direccion "superior" fija, sin releer geometria por-slice.
- **Patient coordinate system**: DICOM PS3.3 C.7.6.2.1.1 fija, para un sujeto **biped** (humano), que
  `+z` apunta siempre hacia la cabeza del paciente -- es una propiedad del sistema de coordenadas del
  paciente completo, no de una imagen individual: ninguna `ImageOrientationPatient`/
  `ImagePositionPatient` de un slice especifico puede "redefinir" cual direccion es superior.
- **`AnatomicalOrientationType`** (0010,2210), leida segun DICOM PS3.3 C.7.6.2.1.1:
  - **ausente, `None`, cadena vacia, o `BIPED` explicito** -> el patient coordinate system humano
    estandar aplica (`+x`=izquierda del paciente, `+y`=posterior, `+z`=cabeza/superior). Un tag
    ausente **NO es geometria insuficiente** -- DICOM define `BIPED` como el comportamiento por
    defecto cuando el tag no esta presente. Se registra explicitamente
    `effective_anatomical_orientation_type="BIPED_DEFAULT"` en ese caso (vs. `"BIPED_EXPLICIT"`
    cuando el tag si estaba presente y decia `BIPED`) -- nunca una asuncion silenciosa.
  - **`QUADRUPED`** -> la convencion +z=superior NO aplica (veterinaria); `ABSTAIN` con
    `reason=UNSUPPORTED_QUADRUPED_ORIENTATION`.
  - **cualquier otro valor no reconocido** -> `ABSTAIN` con
    `reason=UNRECOGNIZED_ANATOMICAL_ORIENTATION_TYPE`.
  - `BIPED` nunca se infiere de resultados del modelo, del smoke, o de RSNA level labels -- viene
    exclusivamente de la semantica del tag DICOM (o de su ausencia, via el default del estandar).
- **Ninguna otra fuente de informacion** se usa para decidir el signo: no RSNA level labels, no GT
  coordinates, no outcome del smoke, no study-specific tuning.

**Direccion canonica de 67B1 (explicita, para no confundir dos convenciones):**
`reference_superior_direction = [0, 0, +1]` es la direccion DICOM fija hacia la cabeza (superior).
Pero el `oriented_axis` que 67B1 usa para ordenar instancias apunta en la direccion OPUESTA --
`canonical_axis_semantics = "SUPERIOR_TO_INFERIOR"` (craneal->caudal, `[0, 0, -1]`) -- porque el
rank predicho debe crecer siguiendo L1-L2 -> L2-L3 -> L3-L4 -> L4-L5 -> L5-S1 (de superior a
inferior). Por eso: eje crudo alineado con `+Z` (superior) -> `FLIP`; eje crudo alineado con `-Z`
(caudal) -> `PRESERVE`.

La unica tolerancia numerica (`NUMERICAL_TOLERANCE`) es de punto flotante -- **no** es un threshold
clinico/anatomico ni fue ajustada mirando resultados.

El GT (`evaluate_direction_against_gt`) se usa **solo despues**, exclusivamente para puntuar el
resultado (SAME/REVERSED/MIXED) -- **nunca** para elegir el signo, ni para decidir si "voltear" una
secuencia ya calculada (patron explicitamente prohibido: `if GT reversed: flip()`).


In [22]:
NUMERICAL_TOLERANCE = 1e-6  # pure floating-point tolerance (near-orthogonality check) -- NOT a
                             # clinical/anatomical threshold, and never tuned by looking at outcomes.

REFERENCE_SUPERIOR_DIRECTION_PATIENT_SPACE = np.array([0.0, 0.0, 1.0])
# DICOM PS3.3 C.7.6.2.1.1 fixes this as a constant of the patient-based coordinate system for a
# BIPED (human) subject: +z always points toward the patient's head, independent of any single
# image's ImageOrientationPatient/ImagePositionPatient (those position/orient a SLICE within this
# fixed patient frame -- they do not redefine which direction is "superior" for the patient as a
# whole). predicted_axis is already expressed in this same patient frame (it is the SVD principal
# axis of centroids computed via dicom_pixel_to_patient_xyz), so it can be compared directly.

CANONICAL_AXIS_SEMANTICS = "SUPERIOR_TO_INFERIOR"  # oriented_axis points cranial(L1-L2)->caudal(L5-S1), i.e. -z -- the OPPOSITE of reference_superior_direction (+z). Never call oriented_axis "the superior direction": that would be the wrong sign.


def classify_anatomical_orientation_type(raw_value) -> dict:
    # DICOM PS3.3 C.7.6.2.1.1: absent/None/empty/"BIPED" -> standard human patient coordinate
    # system applies (BIPED is the documented DEFAULT when the tag is absent, not a fallback we
    # invented). "QUADRUPED" -> unsupported by this rule. Anything else -> unrecognized. This
    # classification comes exclusively from the DICOM tag semantics -- never from model outputs,
    # RSNA labels, or smoke-run outcomes.
    if raw_value is None or str(raw_value).strip() == "":
        return {"effective": "BIPED_DEFAULT", "supported": True}
    normalized = str(raw_value).strip().upper()
    if normalized == "BIPED":
        return {"effective": "BIPED_EXPLICIT", "supported": True}
    if normalized == "QUADRUPED":
        return {"effective": "QUADRUPED", "supported": False, "abstain_reason": "UNSUPPORTED_QUADRUPED_ORIENTATION"}
    return {"effective": f"UNRECOGNIZED:{normalized}", "supported": False, "abstain_reason": "UNRECOGNIZED_ANATOMICAL_ORIENTATION_TYPE"}


def resolve_axis_direction(predicted_axis, anatomical_orientation_type: str = None, geometry_valid: bool = True) -> dict:
    raw_axis = None if predicted_axis is None else np.asarray(predicted_axis, dtype=np.float64)
    reference = REFERENCE_SUPERIOR_DIRECTION_PATIENT_SPACE
    orientation_classification = classify_anatomical_orientation_type(anatomical_orientation_type)

    if not geometry_valid:
        return {
            "status": "ABSTAIN", "oriented_axis": None,
            "raw_axis": raw_axis.tolist() if raw_axis is not None else None,
            "reference_superior_direction": reference.tolist(), "canonical_axis_semantics": CANONICAL_AXIS_SEMANTICS, "axis_alignment": None,
            "anatomical_orientation_type_raw": anatomical_orientation_type, "anatomical_orientation_type_effective": orientation_classification["effective"],
            "reason": "REQUIRED_DICOM_GEOMETRY_UNAVAILABLE: the ImageOrientationPatient/ImagePositionPatient/PixelSpacing that fed predicted_axis did not pass check_geometry_finite/check_orientation_valid.",
        }

    if not orientation_classification["supported"]:
        return {
            "status": "ABSTAIN", "oriented_axis": None,
            "raw_axis": raw_axis.tolist() if raw_axis is not None else None,
            "reference_superior_direction": reference.tolist(), "canonical_axis_semantics": CANONICAL_AXIS_SEMANTICS, "axis_alignment": None,
            "anatomical_orientation_type_raw": anatomical_orientation_type, "anatomical_orientation_type_effective": orientation_classification["effective"],
            "reason": f"{orientation_classification['abstain_reason']}: AnatomicalOrientationType='{anatomical_orientation_type}' -- the +z=superior human convention this rule relies on is not guaranteed, so no defensible orientation can be assigned.",
        }

    if raw_axis is None:
        return {
            "status": "ABSTAIN", "oriented_axis": None, "raw_axis": None,
            "reference_superior_direction": reference.tolist(), "canonical_axis_semantics": CANONICAL_AXIS_SEMANTICS, "axis_alignment": None,
            "anatomical_orientation_type_raw": anatomical_orientation_type, "anatomical_orientation_type_effective": orientation_classification["effective"],
            "reason": "predicted_axis is None (fewer than 2 predicted instances -- spine_axis_from_points could not compute an axis).",
        }

    axis_alignment = float(np.dot(raw_axis, reference))
    if abs(axis_alignment) <= NUMERICAL_TOLERANCE:
        return {
            "status": "ABSTAIN", "oriented_axis": None, "raw_axis": raw_axis.tolist(),
            "reference_superior_direction": reference.tolist(), "canonical_axis_semantics": CANONICAL_AXIS_SEMANTICS, "axis_alignment": axis_alignment,
            "anatomical_orientation_type_raw": anatomical_orientation_type, "anatomical_orientation_type_effective": orientation_classification["effective"],
            "reason": f"predicted_axis is numerically orthogonal to the patient +z axis (within NUMERICAL_TOLERANCE={NUMERICAL_TOLERANCE}, a floating-point tolerance only) -- the cranial/caudal convention cannot disambiguate an in-plane axis.",
        }

    if axis_alignment > 0:
        oriented_axis = -raw_axis
        action_reason = f"dot(raw_axis, reference_superior_direction)={axis_alignment:.6f} > 0 -> raw_axis pointed toward reference_superior_direction (+z), so it was FLIPPED"
    else:
        oriented_axis = raw_axis
        action_reason = f"dot(raw_axis, reference_superior_direction)={axis_alignment:.6f} < 0 -> raw_axis already pointed opposite reference_superior_direction (-z), so it was PRESERVED"

    return {
        "status": "RESOLVED", "oriented_axis": oriented_axis, "raw_axis": raw_axis,
        "reference_superior_direction": reference.tolist(), "canonical_axis_semantics": CANONICAL_AXIS_SEMANTICS, "axis_alignment": axis_alignment,
        "anatomical_orientation_type_raw": anatomical_orientation_type, "anatomical_orientation_type_effective": orientation_classification["effective"],
        "reason": (
            f"DICOM patient coordinate system (PS3.3 C.7.6.2.1.1; AnatomicalOrientationType_effective="
            f"{orientation_classification['effective']}): {action_reason} to point "
            f"{CANONICAL_AXIS_SEMANTICS} (oriented_axis), so ascending projection order is "
            f"cranial(L1-L2) -> caudal(L5-S1). No ground truth, RSNA level label, smoke-run outcome, "
            f"or study-specific tuning was used to make this decision."
        ),
    }


def evaluate_direction_against_gt(oriented_axis, reference_point, gt_level_xyz: dict) -> dict:
    # POST-HOC EVALUATION ONLY -- called strictly after resolve_axis_direction() already fixed the
    # sign. Never used to choose oriented_axis, never used to flip a prediction. gt_level_xyz maps
    # canonical level name -> physical XYZ of that level's ABSOLUTE_LEVEL_REFERENCE_POINT.
    if oriented_axis is None or not gt_level_xyz:
        return {"direction_vs_absolute_GT": "NOT_EVALUATED"}
    levels_present = [lv for lv in CANONICAL_LEVELS if lv in gt_level_xyz]
    if len(levels_present) < 2:
        return {"direction_vs_absolute_GT": "NOT_EVALUATED"}
    positions = project_points_onto_axis([gt_level_xyz[lv] for lv in levels_present], oriented_axis, reference_point)
    order = [lv for _, lv in sorted(zip(positions, levels_present))]
    expected_order = [lv for lv in CANONICAL_LEVELS if lv in levels_present]
    if order == expected_order:
        return {"direction_vs_absolute_GT": "SAME"}
    if order == list(reversed(expected_order)):
        return {"direction_vs_absolute_GT": "REVERSED"}
    return {"direction_vs_absolute_GT": "MIXED"}


# Synthetic self-tests (GT plays NO role in ANY of these):

# A) AnatomicalOrientationType="BIPED" (explicit), raw axis +Z -> resolved, FLIP, oriented -Z.
_r_a = resolve_axis_direction(np.array([0.0, 0.0, 1.0]), anatomical_orientation_type="BIPED")
assert _r_a["status"] == "RESOLVED" and np.allclose(_r_a["oriented_axis"], [0.0, 0.0, -1.0]) and _r_a["anatomical_orientation_type_effective"] == "BIPED_EXPLICIT", "resolve_axis_direction self-test FAILED (case A: explicit BIPED, +Z raw axis should resolve+flip to -Z)"

# B) AnatomicalOrientationType MISSING, raw axis +Z -> resolved via DICOM default BIPED, FLIP, oriented -Z.
_r_b = resolve_axis_direction(np.array([0.0, 0.0, 1.0]), anatomical_orientation_type=None)
assert _r_b["status"] == "RESOLVED" and np.allclose(_r_b["oriented_axis"], [0.0, 0.0, -1.0]) and _r_b["anatomical_orientation_type_effective"] == "BIPED_DEFAULT", "resolve_axis_direction self-test FAILED (case B: missing tag should default to BIPED_DEFAULT, +Z raw axis should resolve+flip to -Z)"

# C) AnatomicalOrientationType MISSING, raw axis -Z -> resolved via DICOM default BIPED, PRESERVE.
_r_c = resolve_axis_direction(np.array([0.0, 0.0, -1.0]), anatomical_orientation_type=None)
assert _r_c["status"] == "RESOLVED" and np.allclose(_r_c["oriented_axis"], [0.0, 0.0, -1.0]) and _r_c["anatomical_orientation_type_effective"] == "BIPED_DEFAULT", "resolve_axis_direction self-test FAILED (case C: missing tag, -Z raw axis should resolve+preserve)"

# D) AnatomicalOrientationType="QUADRUPED" -> ABSTAIN.
_r_d = resolve_axis_direction(np.array([0.0, 0.0, 1.0]), anatomical_orientation_type="QUADRUPED")
assert _r_d["status"] == "ABSTAIN" and "UNSUPPORTED_QUADRUPED_ORIENTATION" in _r_d["reason"], "resolve_axis_direction self-test FAILED (case D: QUADRUPED should ABSTAIN with UNSUPPORTED_QUADRUPED_ORIENTATION)"

# E) AnatomicalOrientationType="SOMETHING_UNKNOWN" -> ABSTAIN.
_r_e = resolve_axis_direction(np.array([0.0, 0.0, 1.0]), anatomical_orientation_type="SOMETHING_UNKNOWN")
assert _r_e["status"] == "ABSTAIN" and "UNRECOGNIZED_ANATOMICAL_ORIENTATION_TYPE" in _r_e["reason"], "resolve_axis_direction self-test FAILED (case E: unrecognized tag value should ABSTAIN with UNRECOGNIZED_ANATOMICAL_ORIENTATION_TYPE)"

# F) invalid/missing required DICOM geometry -> ABSTAIN (covers: no axis, orthogonal axis, and geometry_valid=False).
assert resolve_axis_direction(None)["status"] == "ABSTAIN", "resolve_axis_direction self-test FAILED (case F1: None axis should ABSTAIN)"
assert resolve_axis_direction(np.array([1.0, 0.0, 0.0]))["status"] == "ABSTAIN", "resolve_axis_direction self-test FAILED (case F2: in-plane/orthogonal axis should ABSTAIN)"
assert resolve_axis_direction(np.array([0.0, 0.0, 1.0]), geometry_valid=False)["status"] == "ABSTAIN", "resolve_axis_direction self-test FAILED (case F3: invalid upstream geometry should ABSTAIN)"

# classify_anatomical_orientation_type: direct unit coverage of the DICOM-semantics classification itself.
assert classify_anatomical_orientation_type(None) == {"effective": "BIPED_DEFAULT", "supported": True}, "classify_anatomical_orientation_type self-test FAILED (None)"
assert classify_anatomical_orientation_type("") == {"effective": "BIPED_DEFAULT", "supported": True}, "classify_anatomical_orientation_type self-test FAILED (empty string)"
assert classify_anatomical_orientation_type("BIPED") == {"effective": "BIPED_EXPLICIT", "supported": True}, "classify_anatomical_orientation_type self-test FAILED (BIPED)"
assert classify_anatomical_orientation_type("QUADRUPED")["supported"] is False, "classify_anatomical_orientation_type self-test FAILED (QUADRUPED)"
assert classify_anatomical_orientation_type("WEIRD")["supported"] is False, "classify_anatomical_orientation_type self-test FAILED (unrecognized)"

_gt_same = {"L1-L2": np.array([0.0, 0.0, 20.0]), "L5-S1": np.array([0.0, 0.0, 0.0])}
_eval_same = evaluate_direction_against_gt(np.array([0.0, 0.0, -1.0]), np.array([0.0, 0.0, 20.0]), _gt_same)
assert _eval_same["direction_vs_absolute_GT"] == "SAME", "evaluate_direction_against_gt self-test FAILED (expected SAME)"
print("resolve_axis_direction / classify_anatomical_orientation_type: synthetic self-tests PASSED (A-F, incl. missing-tag=BIPED_DEFAULT, QUADRUPED/unrecognized->ABSTAIN -- GT never consulted).")
print("evaluate_direction_against_gt: synthetic self-test PASSED (post-hoc scoring only).")


resolve_axis_direction / classify_anatomical_orientation_type: synthetic self-tests PASSED (A-F, incl. missing-tag=BIPED_DEFAULT, QUADRUPED/unrecognized->ABSTAIN -- GT never consulted).
evaluate_direction_against_gt: synthetic self-test PASSED (post-hoc scoring only).


## EXPERIMENTO 2 -- Enumerar ventanas contiguas de 5 discos (sin GT)

De N instancias predichas (ya ordenadas craneal->caudal por `oriented_axis`), se enumeran TODAS las
ventanas contiguas de 5 usando solo ranks relativos (window 0 = ranks 1..5, window 1 = ranks 2..6,
..., window N-5 = ranks N-4..N). Nunca se usa GT para elegir cuales ventanas existen, y nunca se
persisten IDs crudos -- solo ranks relativos.


In [23]:
WINDOW_SIZE = 5


def enumerate_candidate_windows(n_instances: int, window_size: int = WINDOW_SIZE) -> list:
    if n_instances < window_size:
        return []
    return [
        {"window_index": w, "rank_start": w + 1, "rank_end": w + window_size, "ranks": list(range(w + 1, w + window_size + 1))}
        for w in range(n_instances - window_size + 1)
    ]


assert enumerate_candidate_windows(4) == [], "enumerate_candidate_windows self-test FAILED (N<5 should yield no windows)"
assert enumerate_candidate_windows(5) == [{"window_index": 0, "rank_start": 1, "rank_end": 5, "ranks": [1, 2, 3, 4, 5]}], "enumerate_candidate_windows self-test FAILED (N=5 should yield exactly 1 window)"
_w7 = enumerate_candidate_windows(7)
assert len(_w7) == 3 and _w7[0]["ranks"] == [1, 2, 3, 4, 5] and _w7[-1]["ranks"] == [3, 4, 5, 6, 7], "enumerate_candidate_windows self-test FAILED (N=7 should yield 3 contiguous windows)"
print("enumerate_candidate_windows: synthetic self-tests PASSED (N<5 empty, N=5 single window, N=7 three contiguous windows).")


enumerate_candidate_windows: synthetic self-tests PASSED (N<5 empty, N=5 single window, N=7 three contiguous windows).


## EXPERIMENTO 3 -- Heuristicas de seleccion de ventana (sin GT, sin entrenar nada aun)

Para cada ventana candidata se calculan CUATRO heuristicas **independientes**, cada una reportada
por separado (`window_rank_by_spacing/support/geometry/fov`) -- **sin ponderacion arbitraria entre
heuristicas**. (E) contexto vertebral/canal se omite: no es derivable directamente de la
segmentacion congelada sin un modelo nuevo (limitacion documentada, no implementada).

Regla de consenso conservadora (predeclarada, deterministica): se selecciona una ventana solo si
las CUATRO heuristicas coinciden en el mismo top-1; en caso contrario, `ABSTAIN_WINDOW`. El
objetivo es determinar si la geometria sola alcanza, antes de entrenar nada.


In [24]:
def _coefficient_of_variation(values: list) -> float:
    arr = np.asarray(values, dtype=np.float64)
    mean = float(np.mean(arr))
    if mean == 0.0:
        return float("inf")
    return float(np.std(arr) / abs(mean))


def window_heuristic_spacing(candidate_windows: list, ordered_instances: list) -> list:
    # (A) Adjacent longitudinal spacing pattern, normalized: lower coefficient of variation across
    # the 4 adjacent gaps in a window = more regular disc-space spacing = better (rank 1).
    scored = []
    for w in candidate_windows:
        members = [ordered_instances[r - 1] for r in w["ranks"]]
        gaps = [abs(members[i + 1]["position_along_axis_mm"] - members[i]["position_along_axis_mm"]) for i in range(len(members) - 1)]
        scored.append((w["window_index"], _coefficient_of_variation(gaps)))
    scored.sort(key=lambda kv: kv[1])
    return [idx for idx, _ in scored]


def window_heuristic_support(candidate_windows: list, ordered_instances: list) -> list:
    # (B) Segmentation support: mean supporting_slice_count + mean instance_confidence, higher is
    # better (rank 1). Combined lexicographically (confidence first, then support), never weighted.
    scored = []
    for w in candidate_windows:
        members = [ordered_instances[r - 1] for r in w["ranks"]]
        mean_confidence = float(np.mean([m["instance_confidence"] for m in members]))
        mean_support = float(np.mean([m["supporting_slice_count"] for m in members]))
        scored.append((w["window_index"], -mean_confidence, -mean_support))
    scored.sort(key=lambda kv: (kv[1], kv[2]))
    return [idx for idx, _, _ in scored]


def window_heuristic_geometry(candidate_windows: list, ordered_instances: list) -> list:
    # (C) Segmentation geometry: prefer no border touch, then lower centroid spread, then larger
    # mean area -- lexicographic ordering (no arbitrary weight combination across sub-criteria).
    scored = []
    for w in candidate_windows:
        members = [ordered_instances[r - 1] for r in w["ranks"]]
        any_border_touch = any(m["border_touch"] for m in members)
        mean_spread = float(np.mean([m["centroid_spread_mm"] for m in members]))
        mean_area = float(np.mean([m["mean_area_px"] for m in members]))
        scored.append((w["window_index"], int(any_border_touch), mean_spread, -mean_area))
    scored.sort(key=lambda kv: (kv[1], kv[2], kv[3]))
    return [idx for idx, _, _, _ in scored]


def window_heuristic_fov(candidate_windows: list, n_instances: int) -> list:
    # (D) FOV geometry: prefer windows with more instances remaining outside the window on both
    # ends (more central within the visible sequence -> less likely truncated by field of view).
    #
    # IMPORTANT (Section 6 audit): FOV position is NOT an anatomical absolute-level anchor. Being
    # centered in the visible sequence is weak circumstantial evidence against truncation -- it is
    # NOT evidence that a window corresponds to L1-S1 specifically. This function deliberately does
    # NOT implement "take the last five visible discs" (or any equivalent edge-anchored rule): such
    # a rule would hardcode 67B's smoke-evidence rank windows (4-8 / 2-6 / 3-7) as a general policy,
    # which is exactly what Section 6 prohibits. This heuristic only ranks candidate windows by how
    # centered they are; it never singles out "the last window" or "the first window" as special.
    scored = []
    for w in candidate_windows:
        margin = min(w["rank_start"] - 1, n_instances - w["rank_end"])
        scored.append((w["window_index"], -margin))
    scored.sort(key=lambda kv: kv[1])
    return [idx for idx, _ in scored]


def select_window_by_consensus(candidate_windows: list, rank_by_spacing: list, rank_by_support: list, rank_by_geometry: list, rank_by_fov: list) -> dict:
    # AGREEMENT is defined, predeclared, exactly as: every AVAILABLE heuristic ranking's top-1
    # choice (index [0]) is the identical window_index. A ranking is NOT_AVAILABLE if it is None or
    # empty. Fixed policy (set before any Colab execution, never adjusted after seeing outcomes):
    # if ANY of the four heuristics is NOT_AVAILABLE, the function abstains COMPLETELY -- it never
    # falls back to a majority/partial consensus of the remaining heuristics, because deciding what
    # counts as "enough" heuristics after the fact would itself be an outcome-driven choice.
    if not candidate_windows:
        return {"selected_window_index": None, "window_selection_status": "ABSTAIN_WINDOW", "window_selection_reason": "INSUFFICIENT_DISC_INSTANCES: fewer than 5 predicted instances, no candidate windows exist.", "heuristics_available": {"spacing": False, "support": False, "geometry": False, "fov": False}}

    rankings = {"spacing": rank_by_spacing, "support": rank_by_support, "geometry": rank_by_geometry, "fov": rank_by_fov}
    heuristics_available = {name: bool(ranking) for name, ranking in rankings.items()}
    if not all(heuristics_available.values()):
        missing = [name for name, ok in heuristics_available.items() if not ok]
        return {"selected_window_index": None, "window_selection_status": "ABSTAIN_WINDOW", "window_selection_reason": f"WINDOW_HEURISTICS_DISAGREE: heuristic(s) {missing} were NOT_AVAILABLE; this notebook's predeclared policy is full abstention, never partial consensus.", "heuristics_available": heuristics_available}

    top_choices = {name: ranking[0] for name, ranking in rankings.items()}
    distinct = set(top_choices.values())
    if len(distinct) == 1:
        return {"selected_window_index": distinct.pop(), "window_selection_status": "SELECTED_UNANIMOUS", "window_selection_reason": "All four independent, available heuristics (spacing, support, geometry, FOV) agree on the same top-ranked window.", "top_choices_by_heuristic": top_choices, "heuristics_available": heuristics_available}
    return {"selected_window_index": None, "window_selection_status": "ABSTAIN_WINDOW", "window_selection_reason": f"WINDOW_HEURISTICS_DISAGREE: heuristics selected different top windows: {top_choices}.", "top_choices_by_heuristic": top_choices, "heuristics_available": heuristics_available}


# Synthetic self-tests: 7 candidate instances -> 3 candidate windows; construct instances so the
# MIDDLE window (index 1, ranks 2..6) is unambiguously best on every heuristic independently.
_syn_instances = []
for _i, _pos in enumerate([0.0, 20.0, 40.0, 60.0, 80.0, 100.0, 200.0]):  # last gap is irregular on purpose
    _syn_instances.append({
        "position_along_axis_mm": _pos, "instance_confidence": 0.9 if 1 <= _i <= 5 else 0.5,
        "supporting_slice_count": 3 if 1 <= _i <= 5 else 1, "border_touch": not (1 <= _i <= 5),
        "centroid_spread_mm": 1.0 if 1 <= _i <= 5 else 10.0, "mean_area_px": 500.0 if 1 <= _i <= 5 else 100.0,
    })
_syn_windows = enumerate_candidate_windows(len(_syn_instances))
assert len(_syn_windows) == 3, "synthetic window setup FAILED (expected 3 candidate windows for N=7)"
_rank_spacing = window_heuristic_spacing(_syn_windows, _syn_instances)
_rank_support = window_heuristic_support(_syn_windows, _syn_instances)
_rank_geometry = window_heuristic_geometry(_syn_windows, _syn_instances)
_rank_fov = window_heuristic_fov(_syn_windows, len(_syn_instances))
assert _rank_support[0] == 1 and _rank_geometry[0] == 1 and _rank_fov[0] == 1, "window heuristics self-test FAILED (expected middle window index=1 to rank first on support/geometry/fov)"
_consensus = select_window_by_consensus(_syn_windows, _rank_spacing, _rank_support, _rank_geometry, _rank_fov)
assert _consensus["window_selection_status"] in ("SELECTED_UNANIMOUS", "ABSTAIN_WINDOW"), "select_window_by_consensus self-test FAILED (unexpected status)"
_empty_consensus = select_window_by_consensus([], [], [], [], [])
assert _empty_consensus["window_selection_status"] == "ABSTAIN_WINDOW", "select_window_by_consensus self-test FAILED (empty candidate list should abstain)"
_not_available_consensus = select_window_by_consensus(_syn_windows, _rank_spacing, _rank_support, _rank_geometry, [])
assert _not_available_consensus["window_selection_status"] == "ABSTAIN_WINDOW" and _not_available_consensus["heuristics_available"]["fov"] is False, "select_window_by_consensus self-test FAILED (NOT_AVAILABLE heuristic (fov=[]) must trigger full abstention, never partial consensus of the other three)"
print("window_heuristic_spacing/support/geometry/fov / select_window_by_consensus: synthetic self-tests PASSED (incl. NOT_AVAILABLE-heuristic full-abstention policy).")


window_heuristic_spacing/support/geometry/fov / select_window_by_consensus: synthetic self-tests PASSED (incl. NOT_AVAILABLE-heuristic full-abstention policy).


## Seccion 11 -- Evaluacion POST-HOC de la ventana seleccionada contra GT (nunca retroalimenta)

El GT se usa aqui SOLO despues de que el algoritmo ya eligio `candidate_window` (o `ABSTAIN_WINDOW`)
usando exclusivamente las heuristicas geometricas del Experimento 3. Calcula `correct_window`,
`selected_window_exact_match`, `GT_window_candidate_rank`, `GT_window_contiguous` -- diagnostico
puro, nunca retroalimenta la seleccion.


In [25]:
def evaluate_window_selection_against_gt(candidate_windows: list, selected_window_index, ordered_instances: list, oriented_axis, reference_point, gt_level_xyz: dict) -> dict:
    if oriented_axis is None or len(gt_level_xyz) < WINDOW_SIZE or not candidate_windows:
        return {"correct_window": None, "selected_window_exact_match": None, "GT_window_candidate_rank": None, "GT_window_contiguous": None}

    levels_present = [lv for lv in CANONICAL_LEVELS if lv in gt_level_xyz]
    if len(levels_present) < WINDOW_SIZE:
        return {"correct_window": None, "selected_window_exact_match": None, "GT_window_candidate_rank": None, "GT_window_contiguous": None}

    gt_positions = project_points_onto_axis([gt_level_xyz[lv] for lv in levels_present], oriented_axis, reference_point)
    instance_positions = [inst["position_along_axis_mm"] for inst in ordered_instances]

    best_window_index, best_total_distance = None, None
    for w in candidate_windows:
        members_positions = [instance_positions[r - 1] for r in w["ranks"]]
        assignment = axis_hungarian_assignment_no_threshold(gt_positions, members_positions)
        total_distance = sum(a["distance_mm"] for a in assignment) if len(assignment) == WINDOW_SIZE else float("inf")
        if best_total_distance is None or total_distance < best_total_distance:
            best_total_distance = total_distance
            best_window_index = w["window_index"]

    correct_window = best_window_index
    ranked_by_fit = sorted(candidate_windows, key=lambda w: sum(
        a["distance_mm"] for a in axis_hungarian_assignment_no_threshold(gt_positions, [instance_positions[r - 1] for r in w["ranks"]])
    ))
    gt_candidate_rank = next((i + 1 for i, w in enumerate(ranked_by_fit) if w["window_index"] == correct_window), None)

    return {
        "correct_window": correct_window,
        "selected_window_exact_match": (selected_window_index is not None and selected_window_index == correct_window),
        "GT_window_candidate_rank": gt_candidate_rank,
        "GT_window_contiguous": True,  # by construction: enumerate_candidate_windows only yields contiguous windows.
    }


# Synthetic self-test: 3 candidate windows (N=7), GT levels correspond exactly to the middle window.
_gt_xyz_syn = {lv: np.array([0.0, 0.0, -20.0 - 20.0 * i]) for i, lv in enumerate(CANONICAL_LEVELS)}
_syn_axis = np.array([0.0, 0.0, -1.0])
_syn_ref = np.array([0.0, 0.0, 0.0])
_syn_instances_11 = [{"position_along_axis_mm": p} for p in [0.0, 20.0, 40.0, 60.0, 80.0, 100.0, 120.0]]
_syn_windows_11 = enumerate_candidate_windows(len(_syn_instances_11))
# reference_point chosen so ascending axis-projection of GT XYZ matches instance positions [20..100] i.e. window index 1
_eval11 = evaluate_window_selection_against_gt(_syn_windows_11, 1, _syn_instances_11, _syn_axis, _syn_ref, _gt_xyz_syn)
assert _eval11["correct_window"] == 1, f"evaluate_window_selection_against_gt self-test FAILED (expected correct_window=1, got {_eval11['correct_window']})"
assert _eval11["selected_window_exact_match"] is True, "evaluate_window_selection_against_gt self-test FAILED (expected exact match)"
print("evaluate_window_selection_against_gt: synthetic self-test PASSED (GT-aligned window correctly identified post-hoc).")


evaluate_window_selection_against_gt: synthetic self-test PASSED (GT-aligned window correctly identified post-hoc).


## EXPERIMENTO 4 -- Combinar direccion + ventana -> secuencia absoluta (`GATE_67B1_C`)

Une el eje orientado (Experimento 1) con la ventana seleccionada (Experimento 3) para producir la
secuencia absoluta de 5 niveles: `rank1 -> L1-L2, rank2 -> L2-L3, ..., rank5 -> L5-S1` dentro de la
ventana elegida. `prediction_status` cubre TODOS los casos, incluyendo abstencion explicita -- nunca
se fuerza una secuencia de 5 niveles si falta evidencia (Seccion 17).


In [26]:
ABSTENTION_REASONS = {
    "INSUFFICIENT_DISC_INSTANCES": "Fewer than 5 predicted disc instances -- no 5-level window can be formed.",
    "AXIS_DIRECTION_UNRESOLVED": "resolve_axis_direction could not fix a cranial/caudal sign (degenerate in-plane axis or no axis at all).",
    "WINDOW_HEURISTICS_DISAGREE": "The four independent window-selection heuristics did not agree on the same top window.",
    "FOV_INCOMPLETE": "The visible sagittal field of view does not appear to contain a full 5-level lumbar window.",
    "REQUIRED_DICOM_GEOMETRY_UNAVAILABLE": "Required DICOM geometry (orientation/position/spacing) could not be resolved for this study.",
}
# NOTE (Section 6/7 audit): FOV_INCOMPLETE is a documented category but this notebook intentionally
# does NOT auto-trigger it via any hardcoded FOV-edge/truncation threshold -- inventing such a
# threshold (e.g. "abstain if fewer than K instances remain outside the window") would itself be
# exactly the kind of un-audited anatomical/clinical threshold Section 6 prohibits. It remains
# available in the abstention-reason vocabulary for a future rule that does not invent one.


def classify_non_executed_frozen_result(run_status: str) -> dict:
    # Used by the orchestration loop below when run_frozen_67b1_on_study did not reach "executed"
    # (no Sagittal T2/STIR series resolved, no DICOM files found, or an insufficient/invalid slice
    # stack) -- i.e. the required DICOM geometry for this study could not be resolved at all.
    return {
        "prediction_status": "INSUFFICIENT_PREDICTED_INSTANCES",
        "abstention_reason": "REQUIRED_DICOM_GEOMETRY_UNAVAILABLE",
        "reason_detail": f"run_frozen_67b1_on_study status={run_status}",
    }


assert classify_non_executed_frozen_result("no_dicom_files_found")["abstention_reason"] == "REQUIRED_DICOM_GEOMETRY_UNAVAILABLE", "classify_non_executed_frozen_result self-test FAILED (case: REQUIRED_DICOM_GEOMETRY_UNAVAILABLE)"
print("classify_non_executed_frozen_result: synthetic self-test PASSED.")


def predict_absolute_level_sequence(predicted_instance_count: int, direction_result: dict, window_result: dict, candidate_windows: list, ordered_instances: list) -> dict:
    if predicted_instance_count < WINDOW_SIZE:
        return {"prediction_status": "INSUFFICIENT_PREDICTED_INSTANCES", "abstention_reason": "INSUFFICIENT_DISC_INSTANCES", "level_sequence": None, "candidate_window": None}

    direction_ok = direction_result.get("status") == "RESOLVED"
    window_ok = window_result.get("window_selection_status") == "SELECTED_UNANIMOUS"

    if not direction_ok and not window_ok:
        return {"prediction_status": "ABSTAIN_BOTH", "abstention_reason": "AXIS_DIRECTION_UNRESOLVED", "level_sequence": None, "candidate_window": window_result.get("selected_window_index")}
    if not direction_ok:
        return {"prediction_status": "ABSTAIN_DIRECTION", "abstention_reason": "AXIS_DIRECTION_UNRESOLVED", "level_sequence": None, "candidate_window": window_result.get("selected_window_index")}
    if not window_ok:
        return {"prediction_status": "ABSTAIN_WINDOW", "abstention_reason": "WINDOW_HEURISTICS_DISAGREE", "level_sequence": None, "candidate_window": None}

    selected_window = next(w for w in candidate_windows if w["window_index"] == window_result["selected_window_index"])
    level_sequence = {lv: rank for lv, rank in zip(CANONICAL_LEVELS, selected_window["ranks"])}
    return {"prediction_status": "ABSOLUTE_SEQUENCE_PREDICTED", "abstention_reason": None, "level_sequence": level_sequence, "candidate_window": selected_window["window_index"]}


def evaluate_absolute_sequence_against_gt(level_sequence, ordered_instances: list, oriented_axis, reference_point, gt_level_xyz: dict) -> dict:
    # POST-HOC ONLY. level_sequence maps canonical level name -> rank (1-indexed) within
    # ordered_instances. Compares the predicted level for each rank against the nearest GT level
    # (by axis-projected position), never used to alter level_sequence itself.
    if level_sequence is None or oriented_axis is None:
        return {"level_wise_correct": {}, "exact_sequence_match": None, "monotonic": None}
    levels_present = [lv for lv in CANONICAL_LEVELS if lv in gt_level_xyz]
    if not levels_present:
        return {"level_wise_correct": {}, "exact_sequence_match": None, "monotonic": None}
    gt_positions = project_points_onto_axis([gt_level_xyz[lv] for lv in levels_present], oriented_axis, reference_point)
    gt_order = [lv for _, lv in sorted(zip(gt_positions, levels_present))]

    predicted_order = sorted(level_sequence.keys(), key=lambda lv: level_sequence[lv])
    level_wise_correct = {lv: (lv in gt_order and gt_order.index(lv) == predicted_order.index(lv)) if lv in gt_order else None for lv in predicted_order}
    exact_sequence_match = predicted_order == gt_order
    ranks = [level_sequence[lv] for lv in predicted_order]
    monotonic = ranks == sorted(ranks)
    return {"level_wise_correct": level_wise_correct, "exact_sequence_match": exact_sequence_match, "monotonic": monotonic}


# Synthetic self-tests:
_pred_insufficient = predict_absolute_level_sequence(3, {"status": "RESOLVED"}, {"window_selection_status": "SELECTED_UNANIMOUS", "selected_window_index": 0}, [], [])
assert _pred_insufficient["prediction_status"] == "INSUFFICIENT_PREDICTED_INSTANCES", "predict_absolute_level_sequence self-test FAILED (N<5 case)"

_pred_abstain_dir = predict_absolute_level_sequence(5, {"status": "ABSTAIN"}, {"window_selection_status": "SELECTED_UNANIMOUS", "selected_window_index": 0}, [{"window_index": 0, "ranks": [1, 2, 3, 4, 5]}], [])
assert _pred_abstain_dir["prediction_status"] == "ABSTAIN_DIRECTION", "predict_absolute_level_sequence self-test FAILED (direction unresolved case)"

_pred_abstain_win = predict_absolute_level_sequence(6, {"status": "RESOLVED"}, {"window_selection_status": "ABSTAIN_WINDOW", "selected_window_index": None}, [{"window_index": 0, "ranks": [1, 2, 3, 4, 5]}, {"window_index": 1, "ranks": [2, 3, 4, 5, 6]}], [])
assert _pred_abstain_win["prediction_status"] == "ABSTAIN_WINDOW", "predict_absolute_level_sequence self-test FAILED (window disagreement case)"

_pred_abstain_both = predict_absolute_level_sequence(5, {"status": "ABSTAIN"}, {"window_selection_status": "ABSTAIN_WINDOW", "selected_window_index": None}, [{"window_index": 0, "ranks": [1, 2, 3, 4, 5]}], [])
assert _pred_abstain_both["prediction_status"] == "ABSTAIN_BOTH", "predict_absolute_level_sequence self-test FAILED (both unresolved case)"

_pred_ok = predict_absolute_level_sequence(5, {"status": "RESOLVED"}, {"window_selection_status": "SELECTED_UNANIMOUS", "selected_window_index": 0}, [{"window_index": 0, "ranks": [1, 2, 3, 4, 5]}], [])
assert _pred_ok["prediction_status"] == "ABSOLUTE_SEQUENCE_PREDICTED" and _pred_ok["level_sequence"] == {lv: r for lv, r in zip(CANONICAL_LEVELS, [1, 2, 3, 4, 5])}, "predict_absolute_level_sequence self-test FAILED (fully resolved case)"

_gt_xyz_syn4 = {lv: np.array([0.0, 0.0, -20.0 - 20.0 * i]) for i, lv in enumerate(CANONICAL_LEVELS)}
_eval4 = evaluate_absolute_sequence_against_gt(_pred_ok["level_sequence"], [], np.array([0.0, 0.0, -1.0]), np.array([0.0, 0.0, 0.0]), _gt_xyz_syn4)
assert _eval4["exact_sequence_match"] is True and _eval4["monotonic"] is True, "evaluate_absolute_sequence_against_gt self-test FAILED (expected exact match on aligned synthetic GT)"
print("predict_absolute_level_sequence / evaluate_absolute_sequence_against_gt: synthetic self-tests PASSED (all abstention branches + fully-resolved case).")


classify_non_executed_frozen_result: synthetic self-test PASSED.
predict_absolute_level_sequence / evaluate_absolute_sequence_against_gt: synthetic self-tests PASSED (all abstention branches + fully-resolved case).


## Seccion 13 -- Estrategia de cohortes

Reutiliza el split determinista `train=1382 / validation=296 / internal_test=297`
(`RSNA_INTERNAL_TEST_LOCKED=True`, `internal_test` nunca se toca). **Stage 1**: los mismos 3 studies
de validation que 67B (paridad de implementacion, ya seleccionados arriba). **Stage 2**: cohorte de
DESARROLLO determinista de 30 studies del split TRAIN (nunca validation completo), seleccionada sin
mirar el resultado del modelo -- el GT se inspecciona solo despues, como diagnostico. **Stage 3**:
`RUN_LOCKED_VALIDATION = False` -- la validacion completa NO se ejecuta automaticamente; solo se
prepara el flag para una fase futura explicitamente aprobada, con direccion/ventana/metricas
congeladas.

**Semantica confirmada (Section 4 audit) para el cohort de desarrollo (TRAIN, n=30):** al ser datos
de development/train, el GT de estos 30 studies **si puede usarse POST-HOC** para: diagnosticar
heuristicas, comparar seleccion de ventana, e incluso para *fijar/ajustar* la regla mientras dura la
fase de desarrollo -- eso es exactamente lo que distingue development de validation. Por eso se
registra explicitamente `heuristic_rules_frozen = False` durante esta fase. Antes de cualquier
validation futura, `heuristic_rules_frozen` debe pasar a `True` explicitamente en codigo/config y
quedar registrado en el summary -- validation NUNCA debe usarse para ajustar heuristicas, pesos,
regla de consenso, regla de direccion, o comportamiento de abstencion (eso seria leakage de
validation hacia el desarrollo del metodo).


In [27]:
DEVELOPMENT_COHORT_SIZE = 30
RUN_LOCKED_VALIDATION = False  # <-- Stage 3: intentionally NOT executed automatically. Requires
                                # explicit future approval, after direction rule + window rule +
                                # metrics are frozen. internal_test remains locked regardless.
HEURISTIC_RULES_FROZEN = False  # <-- Stage 2 (TRAIN dev cohort, n=30): heuristics/consensus rule
                                 # may still be diagnosed/adjusted using this cohort's post-hoc GT.
                                 # MUST be set to True explicitly, in code/config, before Stage 3
                                 # (RUN_LOCKED_VALIDATION=True) is ever run -- validation must never
                                 # be used to tune heuristics/weights/consensus/direction/abstention.

development_cohort_opaque = []
development_cohort_raw = []  # in-memory only, never persisted.

if GATE_A_RSNA_dataset_structure == "PASS" and GATE_C_split_leakage == "PASS" and level_column_present:
    train_ids_this_run = set(real_split["train"]) if "real_split" in dir() else set()
    eligible_dev = set()
    if len(sag_t2_canal_stenosis_reference_rows):
        per_study_status_dev = {}
        for r in sag_t2_canal_stenosis_reference_rows:
            per_study_status_dev.setdefault(r["study_id_opaque"], []).append(r["status"])
        opaque_to_raw_dev = {opaque_id(sid): sid for sid in train_ids_this_run}
        for opaque_sid, statuses in per_study_status_dev.items():
            if opaque_sid in opaque_to_raw_dev and all(s == "ok" for s in statuses) and len(statuses) == len(CANONICAL_LEVELS):
                eligible_dev.add(opaque_to_raw_dev[opaque_sid])
    if eligible_dev:
        # Deterministic, seeded selection -- does NOT inspect model outcome before selecting.
        development_cohort_raw = sorted(list(eligible_dev))
        _dev_rng = np.random.default_rng(2026)
        _shuffled_dev = list(development_cohort_raw)
        _dev_rng.shuffle(_shuffled_dev)
        development_cohort_raw = sorted(_shuffled_dev[:DEVELOPMENT_COHORT_SIZE])
        development_cohort_opaque = [opaque_id(sid) for sid in development_cohort_raw]
    print(f"Development cohort eligible pool size (TRAIN split): {len(eligible_dev)}; selected: {len(development_cohort_opaque)}")
else:
    warnings.append("Development cohort selection NOT_RUN: prerequisites (GATE_A/GATE_C/level schema) not met in this run.")

assert not (RUN_LOCKED_VALIDATION and not HEURISTIC_RULES_FROZEN), "RUN_LOCKED_VALIDATION=True requires HEURISTIC_RULES_FROZEN=True -- validation must never run while heuristics are still tunable."
print("RUN_LOCKED_VALIDATION:", RUN_LOCKED_VALIDATION, "(Stage 3 intentionally not executed automatically)")
print("HEURISTIC_RULES_FROZEN:", HEURISTIC_RULES_FROZEN, "(Stage 2 development cohort may still diagnose/adjust heuristics post-hoc using GT)")
print("RSNA_INTERNAL_TEST_LOCKED:", RSNA_INTERNAL_TEST_LOCKED)


Development cohort eligible pool size (TRAIN split): 1322; selected: 30
RUN_LOCKED_VALIDATION: False (Stage 3 intentionally not executed automatically)
HEURISTIC_RULES_FROZEN: False (Stage 2 development cohort may still diagnose/adjust heuristics post-hoc using GT)
RSNA_INTERNAL_TEST_LOCKED: True


## GT physical XYZ por study (solo para evaluacion POST-HOC, nunca para inferencia)

Resuelve, para un study del cohort de smoke/desarrollo, la posicion fisica real de cada
`ABSOLUTE_LEVEL_REFERENCE_POINT` de Spinal Canal Stenosis (uno de los 5 niveles canonicos). Se llama
UNICAMENTE despues de que la prediccion (direccion + ventana + secuencia) ya fue calculada -- nunca
antes.


In [28]:
def resolve_gt_level_xyz_for_study(study_id_raw: str, train_images_root: Path) -> dict:
    rows_this_study = primary_reference_df[primary_reference_df["study_id"] == study_id_raw] if "primary_reference_df" in dir() else pd.DataFrame()
    result = {}
    for _, row in rows_this_study.iterrows():
        level = row.get("level_raw")
        if level not in CANONICAL_LEVELS:
            continue
        series_id_raw = row["series_id"]
        instance_number = int(row["instance_number"])
        dcm_path = resolve_dicom_instance_path(train_images_root, study_id_raw, series_id_raw, instance_number)
        if dcm_path is None:
            continue
        try:
            ds = pydicom.dcmread(str(dcm_path), stop_before_pixels=True)
        except Exception:
            continue
        required = ["PixelSpacing", "ImagePositionPatient", "ImageOrientationPatient"]
        if not all(hasattr(ds, tag) for tag in required):
            continue
        px_spacing = tuple(float(v) for v in ds.PixelSpacing)
        ipp = tuple(float(v) for v in ds.ImagePositionPatient)
        iop = tuple(float(v) for v in ds.ImageOrientationPatient)
        if not (check_geometry_finite(*px_spacing, *ipp, *iop) and check_orientation_valid(iop)):
            continue
        xyz = dicom_pixel_to_patient_xyz(float(row["x"]), float(row["y"]), px_spacing, ipp, iop)
        if np.all(np.isfinite(xyz)):
            result[level] = xyz
    return result


print("resolve_gt_level_xyz_for_study defined (post-hoc evaluation only -- never called before a prediction already exists).")


resolve_gt_level_xyz_for_study defined (post-hoc evaluation only -- never called before a prediction already exists).


## Ejecucion end-to-end -- Stage 1 (parity hard-stop) -> Stage 2 (Experimentos 1-4), gates `GATE_67B1_A/B/C`

**Orden estricto (Section 14):** primero se ejecuta el runtime congelado SOLO sobre el smoke cohort
(3 studies) y se compara `predicted_instance_count` por study contra `EXPECTED_67B_SMOKE_PARITY`
(mapping opaco). Si `all_smoke_cases_parity_pass` es `False`, la corrida entra en
`STOP_BEFORE_67B1_EXPERIMENTS`: **no se ejecutan** los Experimentos 1-4 (ni sobre smoke ni sobre el
cohort de desarrollo) en esta corrida. Solo si la paridad de Stage 1 pasa completa se procede a
Experimento 1 -> 2 -> 3 -> 4 sobre smoke + desarrollo, evaluando POST-HOC contra GT. Ningun paso de
inferencia mira GT antes de producir su decision. Los gates `GATE_67B1_A/B/C` parten en `NOT_RUN` y,
como maximo, llegan a `PARTIAL` en esta corrida (Stage 1+2 exploratorio) -- `PASS` esta reservado
para una fase de validacion bloqueada (Stage 3, `RUN_LOCKED_VALIDATION=True`) explicitamente
aprobada y no ejecutada aqui.


In [29]:
GATE_67B1_A_AXIS_DIRECTION_WITHOUT_GT = "NOT_RUN"
GATE_67B1_B_LUMBAR_WINDOW_WITHOUT_GT = "NOT_RUN"
GATE_67B1_C_ABSOLUTE_LEVEL_SEQUENCE = "NOT_RUN"
GATE_67B1_D_PRIVACY = "NOT_RUN"

direction_diagnostics_rows = []
window_candidates_rows = []
window_selection_diagnostics_rows = []
absolute_sequence_predictions_rows = []

smoke_parity_result = {"status": "NOT_RUN", "expected": EXPECTED_67B_SMOKE_PARITY, "per_case": [], "all_smoke_cases_parity_pass": None}
experiments_execution_status = "NOT_RUN"  # NOT_RUN | STOP_BEFORE_67B1_EXPERIMENTS | EXECUTED

if RSNA_AVAILABLE and GATE_A_RSNA_dataset_structure == "PASS" and GATE_checkpoint_identity_67b1 == "PASS" and rsna_smoke_cohort_raw:
    # --- Stage 1: smoke-only parity check (implementation parity, NOT experiments) ---
    _smoke_full_results = {}
    for _sid in rsna_smoke_cohort_raw:
        _smoke_full_results[_sid] = run_frozen_67b1_on_study(_sid, train_images_root)

    _per_case = []
    for opaque_sid, expected in EXPECTED_67B_SMOKE_PARITY.items():
        _raw_sid = next((s for s in rsna_smoke_cohort_raw if opaque_id(s) == opaque_sid), None)
        observed_result = _smoke_full_results.get(_raw_sid) if _raw_sid is not None else None
        observed_count = observed_result.get("predicted_instance_count") if observed_result and observed_result.get("status") == "executed" else None
        case_pass = observed_count is not None and observed_count == expected["predicted_instance_count"]
        _per_case.append({
            "study_id_opaque": opaque_sid,
            "expected_predicted_instance_count": expected["predicted_instance_count"],
            "observed_predicted_instance_count": observed_count,
            "parity_status": "PASS" if case_pass else "FAIL",
        })

    all_smoke_cases_parity_pass = bool(_per_case) and all(c["parity_status"] == "PASS" for c in _per_case)
    smoke_parity_result = {"status": "EVALUATED", "expected": EXPECTED_67B_SMOKE_PARITY, "per_case": _per_case, "all_smoke_cases_parity_pass": all_smoke_cases_parity_pass}

    if not all_smoke_cases_parity_pass:
        experiments_execution_status = "STOP_BEFORE_67B1_EXPERIMENTS"
        warnings.append("STOP_BEFORE_67B1_EXPERIMENTS (Section 14 hard stop): Stage-1 smoke parity vs 67B's frozen evidence FAILED for at least one case -- direction/window/absolute-sequence experiments were NOT executed this run.")
    else:
        experiments_execution_status = "EXECUTED"

        # --- Stage 2: Experiments 1-4 over smoke (reusing Stage-1 results, no re-inference) + development cohort ---
        _combined_cohort_raw = list(rsna_smoke_cohort_raw)
        for _sid in development_cohort_raw:
            if _sid not in _combined_cohort_raw:
                _combined_cohort_raw.append(_sid)

        for _sid in _combined_cohort_raw:
            study_opaque = opaque_id(_sid)
            result = _smoke_full_results.get(_sid) if _sid in _smoke_full_results else run_frozen_67b1_on_study(_sid, train_images_root)

            if result.get("status") != "executed":
                direction_diagnostics_rows.append({"study_id_opaque": study_opaque, "raw_axis": None, "oriented_axis": None, "status": "ABSTAIN", "reason": f"run_frozen_67b1_on_study status={result.get('status')}", "predicted_instance_count": 0, "anatomical_orientation_type_raw": None, "anatomical_orientation_type_effective": None})
                _non_executed = classify_non_executed_frozen_result(result.get("status"))
                absolute_sequence_predictions_rows.append({"study_id_opaque": study_opaque, "prediction_status": _non_executed["prediction_status"], "abstention_reason": _non_executed["abstention_reason"], "level_sequence": None, "candidate_window": None})
                continue

            ordered_instances = result["consensus_instances"]
            n_instances = result["predicted_instance_count"]

            direction_result = resolve_axis_direction(result["predicted_axis"], anatomical_orientation_type=result.get("anatomical_orientation_type"))
            gt_level_xyz = resolve_gt_level_xyz_for_study(_sid, train_images_root)

            if direction_result["oriented_axis"] is not None and result["predicted_axis_reference_point"] is not None:
                reoriented_positions = project_points_onto_axis([c["centroid_xyz"] for c in ordered_instances], direction_result["oriented_axis"], result["predicted_axis_reference_point"])
                for c, pos in zip(ordered_instances, reoriented_positions):
                    c["position_along_axis_mm"] = pos
                ordered_instances = sorted(ordered_instances, key=lambda c: c["position_along_axis_mm"])

            direction_eval = evaluate_direction_against_gt(direction_result["oriented_axis"], result["predicted_axis_reference_point"], gt_level_xyz)
            direction_diagnostics_rows.append({
                "study_id_opaque": study_opaque,
                "raw_axis": direction_result["raw_axis"].tolist() if hasattr(direction_result["raw_axis"], "tolist") else direction_result["raw_axis"],
                "oriented_axis": direction_result["oriented_axis"].tolist() if hasattr(direction_result["oriented_axis"], "tolist") else direction_result["oriented_axis"],
                "status": direction_result["status"],
                "reason": direction_result["reason"],
                "predicted_instance_count": n_instances,
                "anatomical_orientation_type_raw": direction_result.get("anatomical_orientation_type_raw"),
                "anatomical_orientation_type_effective": direction_result.get("anatomical_orientation_type_effective"),
                "direction_vs_absolute_GT": direction_eval["direction_vs_absolute_GT"],
            })

            candidate_windows = enumerate_candidate_windows(n_instances)
            for w in candidate_windows:
                window_candidates_rows.append({"study_id_opaque": study_opaque, "window_index": w["window_index"], "rank_start": w["rank_start"], "rank_end": w["rank_end"]})

            if candidate_windows:
                rank_spacing = window_heuristic_spacing(candidate_windows, ordered_instances)
                rank_support = window_heuristic_support(candidate_windows, ordered_instances)
                rank_geometry = window_heuristic_geometry(candidate_windows, ordered_instances)
                rank_fov = window_heuristic_fov(candidate_windows, n_instances)
                window_result = select_window_by_consensus(candidate_windows, rank_spacing, rank_support, rank_geometry, rank_fov)
            else:
                window_result = {"selected_window_index": None, "window_selection_status": "ABSTAIN_WINDOW", "window_selection_reason": "INSUFFICIENT_DISC_INSTANCES"}

            window_eval = evaluate_window_selection_against_gt(candidate_windows, window_result["selected_window_index"], ordered_instances, direction_result["oriented_axis"], result["predicted_axis_reference_point"], gt_level_xyz)
            window_selection_diagnostics_rows.append({
                "study_id_opaque": study_opaque,
                "selected_window_index": window_result["selected_window_index"],
                "window_selection_status": window_result["window_selection_status"],
                "window_selection_reason": window_result["window_selection_reason"],
                **window_eval,
            })

            prediction = predict_absolute_level_sequence(n_instances, direction_result, window_result, candidate_windows, ordered_instances)
            sequence_eval = evaluate_absolute_sequence_against_gt(prediction["level_sequence"], ordered_instances, direction_result["oriented_axis"], result["predicted_axis_reference_point"], gt_level_xyz)
            absolute_sequence_predictions_rows.append({
                "study_id_opaque": study_opaque,
                "prediction_status": prediction["prediction_status"],
                "abstention_reason": prediction["abstention_reason"],
                "level_sequence": prediction["level_sequence"],
                "candidate_window": prediction["candidate_window"],
                **sequence_eval,
            })
else:
    warnings.append("End-to-end 67B1 execution NOT_RUN: RSNA unavailable, checkpoint identity did not PASS, or smoke cohort is empty in this run.")

# --- GATE_67B1_A: axis direction resolvable without GT ---
if not direction_diagnostics_rows:
    GATE_67B1_A_AXIS_DIRECTION_WITHOUT_GT = "NOT_RUN"
else:
    _resolved = [r for r in direction_diagnostics_rows if r["status"] == "RESOLVED"]
    _gt_comparable = [r for r in _resolved if r["direction_vs_absolute_GT"] in ("SAME", "REVERSED", "MIXED")]
    if not _resolved:
        GATE_67B1_A_AXIS_DIRECTION_WITHOUT_GT = "UNRESOLVED"
    elif _gt_comparable and all(r["direction_vs_absolute_GT"] == "SAME" for r in _gt_comparable) and len(_resolved) == len(direction_diagnostics_rows):
        # Capped at PARTIAL: this is Stage 1+2 (smoke + development) evidence only, never the
        # locked validation phase (Stage 3, RUN_LOCKED_VALIDATION=True, not executed here).
        GATE_67B1_A_AXIS_DIRECTION_WITHOUT_GT = "PARTIAL"
    else:
        GATE_67B1_A_AXIS_DIRECTION_WITHOUT_GT = "UNRESOLVED"

# --- GATE_67B1_B: lumbar window selectable without GT ---
if not window_selection_diagnostics_rows:
    GATE_67B1_B_LUMBAR_WINDOW_WITHOUT_GT = "NOT_RUN"
else:
    _selected = [r for r in window_selection_diagnostics_rows if r["window_selection_status"] == "SELECTED_UNANIMOUS"]
    _selected_evaluable = [r for r in _selected if r["selected_window_exact_match"] is not None]
    if not _selected:
        GATE_67B1_B_LUMBAR_WINDOW_WITHOUT_GT = "UNRESOLVED"
    elif _selected_evaluable and all(r["selected_window_exact_match"] for r in _selected_evaluable):
        GATE_67B1_B_LUMBAR_WINDOW_WITHOUT_GT = "PARTIAL"
    else:
        GATE_67B1_B_LUMBAR_WINDOW_WITHOUT_GT = "UNRESOLVED"

# --- GATE_67B1_C: combined absolute level sequence ---
if not absolute_sequence_predictions_rows:
    GATE_67B1_C_ABSOLUTE_LEVEL_SEQUENCE = "NOT_RUN"
else:
    _predicted = [r for r in absolute_sequence_predictions_rows if r["prediction_status"] == "ABSOLUTE_SEQUENCE_PREDICTED"]
    _predicted_evaluable = [r for r in _predicted if r.get("exact_sequence_match") is not None]
    if not _predicted:
        GATE_67B1_C_ABSOLUTE_LEVEL_SEQUENCE = "UNRESOLVED"
    elif _predicted_evaluable and all(r["exact_sequence_match"] for r in _predicted_evaluable):
        GATE_67B1_C_ABSOLUTE_LEVEL_SEQUENCE = "PARTIAL"
    else:
        GATE_67B1_C_ABSOLUTE_LEVEL_SEQUENCE = "UNRESOLVED"

# --- GATE_67B1_D: privacy -- structural check, no raw study_id/series_id in any persisted diagnostics ---
def scan_for_raw_identifier_fields(records: list) -> list:
    violations = []
    for record in records:
        for key in record.keys():
            if key in ("study_id", "series_id") or (key.endswith("_id") and not key.endswith("_id_opaque")):
                violations.append(key)
    return sorted(set(violations))


_all_diagnostic_rows = direction_diagnostics_rows + window_candidates_rows + window_selection_diagnostics_rows + absolute_sequence_predictions_rows + smoke_parity_result.get("per_case", [])
_privacy_violations = scan_for_raw_identifier_fields(_all_diagnostic_rows)
GATE_67B1_D_PRIVACY = "PASS" if not _privacy_violations else "FAIL"
if _privacy_violations:
    warnings.append(f"GATE_67B1_D_PRIVACY FAIL: raw identifier-looking fields found in diagnostics: {_privacy_violations}")

# --- Structural safety clamp (Section 3 audit): GATE_67B1_A/B/C can NEVER read PASS while
# RUN_LOCKED_VALIDATION is False, regardless of how clean the Stage 1+2 results are. This is
# enforced here unconditionally -- not merely by the branch logic above never emitting "PASS" -- so
# that a future edit to the gate-assignment logic above cannot silently introduce an accidental
# PASS on N=3 (or N=33) exploratory evidence.
if RUN_LOCKED_VALIDATION is False:
    for _gate_name in ("GATE_67B1_A_AXIS_DIRECTION_WITHOUT_GT", "GATE_67B1_B_LUMBAR_WINDOW_WITHOUT_GT", "GATE_67B1_C_ABSOLUTE_LEVEL_SEQUENCE"):
        _gate_value = globals()[_gate_name]
        assert _gate_value != "PASS", f"{_gate_name}=PASS while RUN_LOCKED_VALIDATION=False -- this must never happen (Stage 1+2 evidence is capped at PARTIAL)."

print("experiments_execution_status:", experiments_execution_status)
print("smoke_parity_result:", json.dumps(smoke_parity_result, indent=2, default=str))
print("GATE_67B1_A_AXIS_DIRECTION_WITHOUT_GT:", GATE_67B1_A_AXIS_DIRECTION_WITHOUT_GT)
print("GATE_67B1_B_LUMBAR_WINDOW_WITHOUT_GT:", GATE_67B1_B_LUMBAR_WINDOW_WITHOUT_GT)
print("GATE_67B1_C_ABSOLUTE_LEVEL_SEQUENCE:", GATE_67B1_C_ABSOLUTE_LEVEL_SEQUENCE)
print("GATE_67B1_D_PRIVACY:", GATE_67B1_D_PRIVACY)
print(f"Cases: direction={len(direction_diagnostics_rows)} window={len(window_selection_diagnostics_rows)} sequence={len(absolute_sequence_predictions_rows)}")


experiments_execution_status: EXECUTED
smoke_parity_result: {
  "status": "EVALUATED",
  "expected": {
    "4f06df2fd53b": {
      "predicted_instance_count": 8
    },
    "d41a396f20c6": {
      "predicted_instance_count": 9
    },
    "ef2ff5b618cf": {
      "predicted_instance_count": 7
    }
  },
  "per_case": [
    {
      "study_id_opaque": "4f06df2fd53b",
      "expected_predicted_instance_count": 8,
      "observed_predicted_instance_count": 8,
      "parity_status": "PASS"
    },
    {
      "study_id_opaque": "d41a396f20c6",
      "expected_predicted_instance_count": 9,
      "observed_predicted_instance_count": 9,
      "parity_status": "PASS"
    },
    {
      "study_id_opaque": "ef2ff5b618cf",
      "expected_predicted_instance_count": 7,
      "observed_predicted_instance_count": 7,
      "parity_status": "PASS"
    }
  ],
  "all_smoke_cases_parity_pass": true
}
GATE_67B1_A_AXIS_DIRECTION_WITHOUT_GT: UNRESOLVED
GATE_67B1_B_LUMBAR_WINDOW_WITHOUT_GT: UNRESOLVED
GATE_67B1_C

## Metricas agregadas (Experimentos 1/3/4) -- solo evaluacion POST-HOC


In [30]:
direction_metrics = {
    "cases_total": len(direction_diagnostics_rows),
    "cases_resolved": sum(1 for r in direction_diagnostics_rows if r["status"] == "RESOLVED"),
    "cases_unresolved": sum(1 for r in direction_diagnostics_rows if r["status"] == "ABSTAIN"),
    "direction_vs_absolute_GT_counts": {
        status: sum(1 for r in direction_diagnostics_rows if r.get("direction_vs_absolute_GT") == status)
        for status in ("SAME", "REVERSED", "MIXED", "NOT_EVALUATED")
    },
    "orientation_type_explicit_biped_count": sum(1 for r in direction_diagnostics_rows if r.get("anatomical_orientation_type_effective") == "BIPED_EXPLICIT"),
    "orientation_type_default_biped_count": sum(1 for r in direction_diagnostics_rows if r.get("anatomical_orientation_type_effective") == "BIPED_DEFAULT"),
    "orientation_type_quadruped_count": sum(1 for r in direction_diagnostics_rows if r.get("anatomical_orientation_type_effective") == "QUADRUPED"),
    "orientation_type_unrecognized_count": sum(1 for r in direction_diagnostics_rows if isinstance(r.get("anatomical_orientation_type_effective"), str) and r["anatomical_orientation_type_effective"].startswith("UNRECOGNIZED:")),
}

window_metrics = {
    "cases_total": len(window_selection_diagnostics_rows),
    "cases_selected_unanimous": sum(1 for r in window_selection_diagnostics_rows if r["window_selection_status"] == "SELECTED_UNANIMOUS"),
    "cases_abstained": sum(1 for r in window_selection_diagnostics_rows if r["window_selection_status"] == "ABSTAIN_WINDOW"),
    "selected_window_exact_match_rate": (
        sum(1 for r in window_selection_diagnostics_rows if r.get("selected_window_exact_match") is True)
        / max(sum(1 for r in window_selection_diagnostics_rows if r.get("selected_window_exact_match") is not None), 1)
    ) if any(r.get("selected_window_exact_match") is not None for r in window_selection_diagnostics_rows) else None,
}

absolute_sequence_metrics = {
    "studies_attempted": len(absolute_sequence_predictions_rows),
    "studies_predicted": sum(1 for r in absolute_sequence_predictions_rows if r["prediction_status"] == "ABSOLUTE_SEQUENCE_PREDICTED"),
    "studies_abstained": sum(1 for r in absolute_sequence_predictions_rows if r["prediction_status"].startswith("ABSTAIN")),
    "studies_insufficient_instances": sum(1 for r in absolute_sequence_predictions_rows if r["prediction_status"] == "INSUFFICIENT_PREDICTED_INSTANCES"),
    "coverage": (
        sum(1 for r in absolute_sequence_predictions_rows if r["prediction_status"] == "ABSOLUTE_SEQUENCE_PREDICTED") / max(len(absolute_sequence_predictions_rows), 1)
    ),
    "exact_sequence_match_rate": (
        sum(1 for r in absolute_sequence_predictions_rows if r.get("exact_sequence_match") is True)
        / max(sum(1 for r in absolute_sequence_predictions_rows if r.get("exact_sequence_match") is not None), 1)
    ) if any(r.get("exact_sequence_match") is not None for r in absolute_sequence_predictions_rows) else None,
    "monotonic_sequence_rate": (
        sum(1 for r in absolute_sequence_predictions_rows if r.get("monotonic") is True)
        / max(sum(1 for r in absolute_sequence_predictions_rows if r.get("monotonic") is not None), 1)
    ) if any(r.get("monotonic") is not None for r in absolute_sequence_predictions_rows) else None,
    "prediction_status_counts": {
        status: sum(1 for r in absolute_sequence_predictions_rows if r["prediction_status"] == status)
        for status in ("ABSOLUTE_SEQUENCE_PREDICTED", "ABSTAIN_DIRECTION", "ABSTAIN_WINDOW", "ABSTAIN_BOTH", "INSUFFICIENT_PREDICTED_INSTANCES")
    },
}

abstention_metrics = {
    "abstention_reason_counts": {
        reason: sum(1 for r in absolute_sequence_predictions_rows if r.get("abstention_reason") == reason)
        for reason in ABSTENTION_REASONS.keys()
    },
    "total_abstained": sum(1 for r in absolute_sequence_predictions_rows if r["prediction_status"] != "ABSOLUTE_SEQUENCE_PREDICTED"),
}

print("direction_metrics:", json.dumps(direction_metrics, indent=2, default=str))
print("window_metrics:", json.dumps(window_metrics, indent=2, default=str))
print("absolute_sequence_metrics:", json.dumps(absolute_sequence_metrics, indent=2, default=str))
print("abstention_metrics:", json.dumps(abstention_metrics, indent=2, default=str))


direction_metrics: {
  "cases_total": 33,
  "cases_resolved": 33,
  "cases_unresolved": 0,
  "direction_vs_absolute_GT_counts": {
    "SAME": 0,
    "REVERSED": 0,
    "MIXED": 0,
    "NOT_EVALUATED": 33
  },
  "orientation_type_explicit_biped_count": 0,
  "orientation_type_default_biped_count": 33,
  "orientation_type_quadruped_count": 0,
  "orientation_type_unrecognized_count": 0
}
window_metrics: {
  "cases_total": 33,
  "cases_selected_unanimous": 3,
  "cases_abstained": 30,
  "selected_window_exact_match_rate": null
}
absolute_sequence_metrics: {
  "studies_attempted": 33,
  "studies_predicted": 3,
  "studies_abstained": 30,
  "studies_insufficient_instances": 0,
  "coverage": 0.09090909090909091,
  "exact_sequence_match_rate": null,
  "monotonic_sequence_rate": null,
  "prediction_status_counts": {
    "ABSOLUTE_SEQUENCE_PREDICTED": 3,
    "ABSTAIN_DIRECTION": 0,
    "ABSTAIN_WINDOW": 30,
    "ABSTAIN_BOTH": 0,
    "INSUFFICIENT_PREDICTED_INSTANCES": 0
  }
}
abstention_metrics: {

## Seccion 18/19 -- Artefactos (`artifacts/post_e50/absolute_level_orientation_window/`)

Todos los study IDs son hashes opacos; nunca se persiste `study_id`/`series_id` crudo en ningun
artefacto.


In [31]:
ANCHOR_DIR.mkdir(parents=True, exist_ok=True)

pfi_67b1_environment = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "git_branch": GIT_BRANCH, "git_commit": GIT_COMMIT, "base_commit_67b": BASE_COMMIT_67B,
    "python_version": sys.version, "torch_version": getattr(torch, "__version__", None),
    "rsna_available": RSNA_AVAILABLE, "official_test_present": OFFICIAL_TEST_PRESENT, "official_test_accessed": OFFICIAL_TEST_ACCESSED,
}
safe_write_text(ANCHOR_DIR / "67B1_environment.json", json.dumps(pfi_67b1_environment, indent=2, default=str))

pfi_67b1_quality_gates = {
    "GATE_A_RSNA_dataset_structure": GATE_A_RSNA_dataset_structure,
    "GATE_C_split_leakage": GATE_C_split_leakage,
    "GATE_D_series_annotation_mapping": GATE_D_series_annotation_mapping,
    "GATE_E_coordinate_physical_parity_real": GATE_E_coordinate_physical_parity_real,
    "GATE_checkpoint_identity_67b1": GATE_checkpoint_identity_67b1,
    "GATE_67B1_A_AXIS_DIRECTION_WITHOUT_GT": GATE_67B1_A_AXIS_DIRECTION_WITHOUT_GT,
    "GATE_67B1_B_LUMBAR_WINDOW_WITHOUT_GT": GATE_67B1_B_LUMBAR_WINDOW_WITHOUT_GT,
    "GATE_67B1_C_ABSOLUTE_LEVEL_SEQUENCE": GATE_67B1_C_ABSOLUTE_LEVEL_SEQUENCE,
    "GATE_67B1_D_PRIVACY": GATE_67B1_D_PRIVACY,
}
safe_write_text(ANCHOR_DIR / "67B1_quality_gates.json", json.dumps(pfi_67b1_quality_gates, indent=2, default=str))
safe_write_text(ANCHOR_DIR / "67B1_smoke_parity.json", json.dumps(smoke_parity_result, indent=2, default=str))

def _rows_to_csv_text(rows: list) -> str:
    if not rows:
        return ""
    frame = pd.DataFrame(rows)
    return frame.to_csv(index=False)

safe_write_text(ANCHOR_DIR / "67B1_direction_diagnostics.csv", _rows_to_csv_text(direction_diagnostics_rows))
safe_write_text(ANCHOR_DIR / "67B1_window_candidates.csv", _rows_to_csv_text(window_candidates_rows))
safe_write_text(ANCHOR_DIR / "67B1_window_selection_diagnostics.csv", _rows_to_csv_text(window_selection_diagnostics_rows))
safe_write_text(ANCHOR_DIR / "67B1_absolute_sequence_predictions.csv", _rows_to_csv_text(absolute_sequence_predictions_rows))

print("Artifacts written under:", ANCHOR_DIR)
for _f in sorted(ANCHOR_DIR.glob("67B1_*")):
    print(" -", _f.name)


Artifacts written under: /content/pfi_post_e50_67B1_workspace/artifacts/post_e50/absolute_level_orientation_window
 - 67B1_absolute_sequence_predictions.csv
 - 67B1_direction_diagnostics.csv
 - 67B1_environment.json
 - 67B1_quality_gates.json
 - 67B1_smoke_parity.json
 - 67B1_window_candidates.csv
 - 67B1_window_selection_diagnostics.csv


## Seccion 20 -- Decision (nunca `ABSOLUTE_LEVEL_ANCHOR_VALIDATED` sin validacion bloqueada)

`decision` inicial es `UNRESOLVED`; puede pasar a `PARTIAL` solo con evidencia de smoke/desarrollo
(nunca `VALIDATED`). `AUTOMATIC_DISC_LOCALIZATION_VALIDATED` permanece `False` -- 67B1 sigue siendo
research-only.


In [32]:
AUTOMATIC_DISC_LOCALIZATION_VALIDATED = False  # unchanged -- 67B1 is research-only, never touches product runtime semantics.

_gate_values = [
    GATE_67B1_A_AXIS_DIRECTION_WITHOUT_GT,
    GATE_67B1_B_LUMBAR_WINDOW_WITHOUT_GT,
    GATE_67B1_C_ABSOLUTE_LEVEL_SEQUENCE,
]

if all(v == "NOT_RUN" for v in _gate_values):
    decision = "UNRESOLVED"
elif any(v == "UNRESOLVED" for v in _gate_values):
    decision = "UNRESOLVED"
elif all(v == "PARTIAL" for v in _gate_values):
    decision = "PARTIAL"
else:
    decision = "UNRESOLVED"

# decision is NEVER set to "ABSOLUTE_LEVEL_ANCHOR_VALIDATED" anywhere in this notebook -- that
# outcome is reserved for an explicitly-approved future locked-validation phase (Stage 3,
# RUN_LOCKED_VALIDATION=True), which this notebook intentionally does not execute.

print("decision:", decision)
print("AUTOMATIC_DISC_LOCALIZATION_VALIDATED:", AUTOMATIC_DISC_LOCALIZATION_VALIDATED, "(unchanged)")


decision: UNRESOLVED
AUTOMATIC_DISC_LOCALIZATION_VALIDATED: False (unchanged)


## `67B1_summary.json` (Seccion 19)


In [33]:
pfi_67b1_summary = {
    "notebook": "67B1_postE50_absolute_level_orientation_window",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "base_commit": "351d9a5ba63b3cfffea91180c0075f778a858e20",
    "frozen_checkpoint_sha256": EXPECTED_CHECKPOINT_SHA256,
    "training_performed": False,
    "internal_test_locked": RSNA_INTERNAL_TEST_LOCKED,
    "official_test_accessed": OFFICIAL_TEST_ACCESSED,
    "67b_closing_evidence_reused": REUSE_VALIDATED_67B_GEOMETRY_EVIDENCE and geometry_evidence_source == "REUSED_67B_REAL_COLAB_EVIDENCE",
    "geometry_evidence_source": geometry_evidence_source,
    "smoke_parity": smoke_parity_result,
    "experiments_execution_status": experiments_execution_status,
    "heuristic_rules_frozen": HEURISTIC_RULES_FROZEN,
    "run_locked_validation": RUN_LOCKED_VALIDATION,
    "direction_method": "resolve_axis_direction (DICOM patient-space +z=superior convention; sign never chosen from GT)",
    "window_method": "enumerate_candidate_windows + 4 independent geometric heuristics (spacing/support/geometry/fov) + unanimous-consensus rule; ABSTAIN_WINDOW otherwise",
    "direction_metrics": direction_metrics,
    "window_metrics": window_metrics,
    "absolute_sequence_metrics": absolute_sequence_metrics,
    "abstention_metrics": abstention_metrics,
    "gates": pfi_67b1_quality_gates,
    "decision": decision,
    "warnings": warnings,
    "limitations": [
        "Heuristic (E) vertebral/canal context was NOT implemented: not directly derivable from the frozen segmentation without a new model.",
        "GATE_67B1_A/B/C are capped at PARTIAL in this notebook -- PASS requires an explicitly-approved future locked-validation phase (Stage 3, RUN_LOCKED_VALIDATION=True), not executed here.",
        "Development cohort is 30 TRAIN-split studies (deterministic, seed=2026) -- not the full 1382-study TRAIN split, and never the validation/internal_test splits used for locked evaluation.",
        "No supervised scorer was trained (Sections 15/16 fallback intentionally not implemented) -- this notebook only measures whether pure geometry suffices.",
    ],
}

safe_write_text(ANCHOR_DIR / "67B1_summary.json", json.dumps(pfi_67b1_summary, indent=2, default=str))
print(json.dumps(pfi_67b1_summary, indent=2, default=str))


{
  "notebook": "67B1_postE50_absolute_level_orientation_window",
  "generated_at": "2026-08-20T11:56:39.144199+00:00",
  "base_commit": "351d9a5ba63b3cfffea91180c0075f778a858e20",
  "frozen_checkpoint_sha256": "cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944",
  "training_performed": false,
  "internal_test_locked": true,
  "official_test_accessed": false,
  "67b_closing_evidence_reused": true,
  "geometry_evidence_source": "REUSED_67B_REAL_COLAB_EVIDENCE",
  "smoke_parity": {
    "status": "EVALUATED",
    "expected": {
      "4f06df2fd53b": {
        "predicted_instance_count": 8
      },
      "d41a396f20c6": {
        "predicted_instance_count": 9
      },
      "ef2ff5b618cf": {
        "predicted_instance_count": 7
      }
    },
    "per_case": [
      {
        "study_id_opaque": "4f06df2fd53b",
        "expected_predicted_instance_count": 8,
        "observed_predicted_instance_count": 8,
        "parity_status": "PASS"
      },
      {
        "study_id_opaq

## Seccion 21 -- Reporte (`reports/post_e50/post_e50_absolute_level_orientation_window_report.md`)

Responde EXACTAMENTE Q1-Q5, nunca mas alla de la evidencia disponible en esta corrida.


In [34]:
def _answer_q1() -> str:
    if GATE_67B1_A_AXIS_DIRECTION_WITHOUT_GT == "NOT_RUN":
        return "NOT_RUN in this environment (RSNA data unavailable locally) -- see `warnings` for details."
    if GATE_67B1_A_AXIS_DIRECTION_WITHOUT_GT == "PARTIAL":
        return (
            f"On the Stage 1+2 cohort available this run ({direction_metrics['cases_total']} studies), the DICOM "
            f"patient-space (+z=superior) hypothesis resolved direction for {direction_metrics['cases_resolved']}/"
            f"{direction_metrics['cases_total']} cases, and where GT was comparable it matched SAME in "
            f"{direction_metrics['direction_vs_absolute_GT_counts']['SAME']} case(s), REVERSED in "
            f"{direction_metrics['direction_vs_absolute_GT_counts']['REVERSED']}, MIXED in "
            f"{direction_metrics['direction_vs_absolute_GT_counts']['MIXED']}. This is Stage 1+2 evidence only "
            f"(PARTIAL) -- not a locked-validation PASS."
        )
    return "UNRESOLVED: direction could not be resolved without GT for at least one attempted case this run."


def _answer_q2() -> str:
    if GATE_67B1_B_LUMBAR_WINDOW_WITHOUT_GT == "NOT_RUN":
        return "NOT_RUN in this environment (RSNA data unavailable locally) -- see `warnings` for details."
    if GATE_67B1_B_LUMBAR_WINDOW_WITHOUT_GT == "PARTIAL":
        return (
            f"On the Stage 1+2 cohort, {window_metrics['cases_selected_unanimous']}/{window_metrics['cases_total']} "
            f"cases reached a unanimous 4-heuristic consensus window (the rest abstained: "
            f"{window_metrics['cases_abstained']}), with a selected-window exact-match rate vs GT of "
            f"{window_metrics['selected_window_exact_match_rate']}. PARTIAL Stage 1+2 evidence only."
        )
    return "UNRESOLVED: window heuristics disagreed (or evidence was insufficient) for at least one attempted case this run."


def _answer_q3() -> str:
    if GATE_67B1_C_ABSOLUTE_LEVEL_SEQUENCE == "NOT_RUN":
        return "NOT_RUN in this environment (RSNA data unavailable locally) -- see `warnings` for details."
    return (
        f"Coverage (studies with a full ABSOLUTE_SEQUENCE_PREDICTED, no forced abstention): "
        f"{absolute_sequence_metrics['coverage']:.3f}. Exact five-level sequence match rate vs RSNA GT (where "
        f"comparable): {absolute_sequence_metrics['exact_sequence_match_rate']}. Monotonic sequence rate: "
        f"{absolute_sequence_metrics['monotonic_sequence_rate']}. See `67B1_absolute_sequence_predictions.csv` "
        f"for per-study detail."
    )


def _answer_q4() -> str:
    return (
        f"Yes -- abstention is a first-class outcome, never bypassed. Prediction status breakdown this run: "
        f"{absolute_sequence_metrics['prediction_status_counts']}. Abstention reason breakdown: "
        f"{abstention_metrics['abstention_reason_counts']}."
    )


def _answer_q5() -> str:
    if GATE_67B1_C_ABSOLUTE_LEVEL_SEQUENCE == "NOT_RUN":
        return "Cannot be answered yet in this environment -- no RSNA execution occurred this run. Requires a real Colab run on Stage 1+2 (and eventually Stage 3) before this question is answerable."
    match_rate = absolute_sequence_metrics.get("exact_sequence_match_rate")
    if match_rate is None:
        return "Insufficient GT-comparable cases this run to judge whether geometry alone suffices."
    if match_rate >= 0.95 and absolute_sequence_metrics.get("coverage", 0.0) >= 0.5:
        return f"Preliminary evidence (Stage 1+2 only, n={absolute_sequence_metrics['studies_attempted']}) suggests pure geometry MAY suffice (match_rate={match_rate:.3f}) -- but this must be confirmed on Stage 3 locked validation before any supervised scorer decision is made."
    return f"Preliminary evidence (Stage 1+2 only, n={absolute_sequence_metrics['studies_attempted']}) does not yet show geometry alone is sufficient (match_rate={match_rate:.3f}) -- Sections 15/16's tiny supervised-scorer fallback remains a documented (not implemented) option, contingent on further Stage 1+2/3 evidence."


report_lines = [
    "# Post-E50 -- 67B1 Absolute Level Orientation + Lumbar Window",
    "",
    f"- base_commit (67B closing): `351d9a5ba63b3cfffea91180c0075f778a858e20`",
    f"- frozen_checkpoint_sha256: `{EXPECTED_CHECKPOINT_SHA256}`",
    f"- decision: **{decision}**",
    f"- training_performed: False | internal_test_locked: {RSNA_INTERNAL_TEST_LOCKED} | official_test_accessed: {OFFICIAL_TEST_ACCESSED}",
    f"- AUTOMATIC_DISC_LOCALIZATION_VALIDATED: {AUTOMATIC_DISC_LOCALIZATION_VALIDATED} (unchanged)",
    "",
    "## Gates",
    "```json",
    json.dumps(pfi_67b1_quality_gates, indent=2, default=str),
    "```",
    "",
    "## Q1 -- Is orientation (cranial/caudal) resolvable without GT?",
    _answer_q1(),
    "",
    "## Q2 -- Is the lumbar 5-disc window selectable without GT via geometry alone?",
    _answer_q2(),
    "",
    "## Q3 -- What is the combined direction+window match rate vs RSNA GT?",
    _answer_q3(),
    "",
    "## Q4 -- Does the system abstain instead of forcing a wrong answer?",
    _answer_q4(),
    "",
    "## Q5 -- Is a supervised scorer actually necessary, or does geometry already suffice?",
    _answer_q5(),
    "",
    "## Warnings",
    "\n".join(f"- {w}" for w in warnings) if warnings else "(none)",
    "",
    "## Limitations",
    "\n".join(f"- {l}" for l in pfi_67b1_summary["limitations"]),
]

report_text = "\n".join(report_lines)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
report_path = REPORT_DIR / "post_e50_absolute_level_orientation_window_report.md"
safe_write_text(report_path, report_text)
print(report_text)


# Post-E50 -- 67B1 Absolute Level Orientation + Lumbar Window

- base_commit (67B closing): `351d9a5ba63b3cfffea91180c0075f778a858e20`
- frozen_checkpoint_sha256: `cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944`
- decision: **UNRESOLVED**
- training_performed: False | internal_test_locked: True | official_test_accessed: False
- AUTOMATIC_DISC_LOCALIZATION_VALIDATED: False (unchanged)

## Gates
```json
{
  "GATE_A_RSNA_dataset_structure": "PASS",
  "GATE_C_split_leakage": "PASS",
  "GATE_D_series_annotation_mapping": "PASS",
  "GATE_E_coordinate_physical_parity_real": "PASS",
  "GATE_checkpoint_identity_67b1": "PASS",
  "GATE_67B1_A_AXIS_DIRECTION_WITHOUT_GT": "UNRESOLVED",
  "GATE_67B1_B_LUMBAR_WINDOW_WITHOUT_GT": "UNRESOLVED",
  "GATE_67B1_C_ABSOLUTE_LEVEL_SEQUENCE": "UNRESOLVED",
  "GATE_67B1_D_PRIVACY": "PASS"
}
```

## Q1 -- Is orientation (cranial/caudal) resolvable without GT?
UNRESOLVED: direction could not be resolved without GT for at least one attempted case

## EXPERIMENT STATUS


In [35]:
experiment_status = {
    "Experiment 1 (direction resolution without GT)": "IMPLEMENTED" if "resolve_axis_direction" in dir() else "NOT_IMPLEMENTED",
    "Experiment 2 (contiguous window enumeration)": "IMPLEMENTED" if "enumerate_candidate_windows" in dir() else "NOT_IMPLEMENTED",
    "Experiment 3 (window heuristics + consensus)": "IMPLEMENTED" if "select_window_by_consensus" in dir() else "NOT_IMPLEMENTED",
    "Section 11 (post-hoc GT window evaluation)": "IMPLEMENTED" if "evaluate_window_selection_against_gt" in dir() else "NOT_IMPLEMENTED",
    "Experiment 4 (combined absolute sequence + abstention)": "IMPLEMENTED" if "predict_absolute_level_sequence" in dir() else "NOT_IMPLEMENTED",
    "Section 15 fallback (tiny supervised window scorer)": "NOT_IMPLEMENTED (documented only, per explicit instruction)",
    "Section 16 fallback (tiny supervised orientation scorer)": "NOT_IMPLEMENTED (documented only, per explicit instruction)",
    "Stage 3 locked validation (RUN_LOCKED_VALIDATION)": f"NOT_EXECUTED (flag={RUN_LOCKED_VALIDATION})",
    "Supervised model trained this run": False,
    "internal_test accessed this run": False,
    "official RSNA test accessed this run": OFFICIAL_TEST_ACCESSED,
}
print(json.dumps(experiment_status, indent=2, default=str))


{
  "Experiment 1 (direction resolution without GT)": "IMPLEMENTED",
  "Experiment 2 (contiguous window enumeration)": "IMPLEMENTED",
  "Experiment 3 (window heuristics + consensus)": "IMPLEMENTED",
  "Section 11 (post-hoc GT window evaluation)": "IMPLEMENTED",
  "Experiment 4 (combined absolute sequence + abstention)": "IMPLEMENTED",
  "Section 15 fallback (tiny supervised window scorer)": "NOT_IMPLEMENTED (documented only, per explicit instruction)",
  "Section 16 fallback (tiny supervised orientation scorer)": "NOT_IMPLEMENTED (documented only, per explicit instruction)",
  "Stage 3 locked validation (RUN_LOCKED_VALIDATION)": "NOT_EXECUTED (flag=False)",
  "Supervised model trained this run": false,
  "internal_test accessed this run": false,
  "official RSNA test accessed this run": false
}


## Seccion 22 -- Busquedas estaticas obligatorias (privacidad, scope, no-training)

Cada ocurrencia legitima se explica: strings dentro de mensajes/comentarios/nombres de
funcion/variables (no llamadas reales) son legitimas; una llamada real a `.train()` sobre el modelo,
un `optimizer`, un `backward()`, o un acceso real a `internal_test`/`test_images`/
`sample_submission` NO lo serian.


In [36]:
_this_notebook_source = "\n".join(c["source"] if isinstance(c["source"], str) else "".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code") if "nb" in dir() else None

_static_search_terms = ["internal_test", "test_images", "sample_submission", ".train(", "optimizer", "backward(", "loss.backward"]

if _this_notebook_source is not None:
    for term in _static_search_terms:
        count = _this_notebook_source.count(term)
        print(f"'{term}': {count} occurrence(s)")
else:
    print("Static search over notebook source not available in this generator-script context (no in-memory `nb` object) -- run this cell from the saved .ipynb, or use the external grep-based Section 22 audit described in the closing report.")

print()
print("Explanation of legitimate occurrences (fixed, not derived at runtime):")
print("- 'internal_test': appears only as a split-name string (real_split['internal_test'], RSNA_INTERNAL_TEST_LOCKED) and in comments/warnings explaining that it is never read; never used as a live DataFrame filter or file path.")
print("- '.train(': model is only ever set to .eval() (sagittal_model_67b1.eval()); no '.train()' call exists on any model object.")
print("- 'optimizer' / 'backward(' / 'loss.backward': none defined or called anywhere -- no training loop exists in this notebook.")
print("- 'test_images' / 'sample_submission': RSNA official-test markers, checked only for absence/detection (RSNA_OFFICIAL_TEST_MARKERS, _rsna_content_report) -- never opened or read.")
print("- raw study_id/series_id: exist only transiently in-memory (e.g. rsna_smoke_cohort_raw, development_cohort_raw, _combined_cohort_raw) to resolve real DICOM paths at runtime; GATE_67B1_D_PRIVACY structurally confirms none reach any persisted artifact.")


Static search over notebook source not available in this generator-script context (no in-memory `nb` object) -- run this cell from the saved .ipynb, or use the external grep-based Section 22 audit described in the closing report.

Explanation of legitimate occurrences (fixed, not derived at runtime):
- 'internal_test': appears only as a split-name string (real_split['internal_test'], RSNA_INTERNAL_TEST_LOCKED) and in comments/warnings explaining that it is never read; never used as a live DataFrame filter or file path.
- '.train(': model is only ever set to .eval() (sagittal_model_67b1.eval()); no '.train()' call exists on any model object.
- 'optimizer' / 'backward(' / 'loss.backward': none defined or called anywhere -- no training loop exists in this notebook.
- 'test_images' / 'sample_submission': RSNA official-test markers, checked only for absence/detection (RSNA_OFFICIAL_TEST_MARKERS, _rsna_content_report) -- never opened or read.
- raw study_id/series_id: exist only transiently 

## Seccion 24 -- Estado final de git (NO commit, NO push -- esperar revision)


In [37]:
import subprocess as _subprocess


def _run_git_report(args: list) -> str:
    result = _subprocess.run(["git", *args], cwd=str(REPO_ROOT), capture_output=True, text=True)
    return result.stdout.strip()


_git_branch_final = _run_git_report(["branch", "--show-current"])
_git_head_final = _run_git_report(["rev-parse", "HEAD"])
_git_status_final = _run_git_report(["status", "--short"])
_git_diff_check_final = _run_git_report(["diff", "--check"])
_git_diff_stat_final = _run_git_report(["diff", "--stat"])
_git_diff_names_final = _run_git_report(["diff", "--name-status"])

print("branch:", _git_branch_final)
print("HEAD:", _git_head_final)
print("base_commit_67b (expected ancestor):", BASE_COMMIT_67B)
print()
print("git status --short:")
print(_git_status_final if _git_status_final else "(clean)")
print()
print("git diff --check:")
print(_git_diff_check_final if _git_diff_check_final else "(no whitespace/conflict-marker issues)")
print()
print("git diff --stat:")
print(_git_diff_stat_final if _git_diff_stat_final else "(no diff)")
print()
print("git diff --name-status:")
print(_git_diff_names_final if _git_diff_names_final else "(no diff)")
print()
print("67B files modified:", "notebooks/post_e50/67B_postE50_absolute_level_anchor_rsna.ipynb" in _git_diff_names_final)
print("main branch touched: N/A (this cell only inspects the working tree of the current checkout).")
print()
print("REMINDER: this notebook performs NO git add / commit / push. Review is required before any git action.")


branch: 
HEAD: 
base_commit_67b (expected ancestor): 351d9a5ba63b3cfffea91180c0075f778a858e20

git status --short:
(clean)

git diff --check:
(no whitespace/conflict-marker issues)

git diff --stat:
(no diff)

git diff --name-status:
(no diff)

67B files modified: False
main branch touched: N/A (this cell only inspects the working tree of the current checkout).

REMINDER: this notebook performs NO git add / commit / push. Review is required before any git action.
